<a href="https://colab.research.google.com/github/Japr2206/DSI/blob/main/Action_Geom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# GEOMETRY OF LEARNING — SECTION 7
# UNIFIED FINAL COMPUTATIONAL BLOCK
# THEORY-ALIGNED CORE + VALIDATION LAYER
# ------------------------------------------------------------
# Full Python script
# Fully commented in English
#
# Conceptual principle
# --------------------
# Sections 2–6 are preserved computationally through:
#   - trajectories
#   - local jets / windows
#   - admissible observables
#   - structural profiles
#   - structural masses
#   - superlevels and boundaries
#
# The macroscopic event logic is sharpened via a multiscale
# structural boundary score.
#
# Validation principle
# --------------------
# The script closes Section 7 in a single executable block:
#   (A) global geometry
#   (B) structural profiles
#   (C) events / regimes / superlevels
#   (D) multi-family discrimination
#   (E) temporal null control
#   (F) parser robustness
# ============================================================

import numpy as np
import pandas as pd
import warnings
from copy import deepcopy

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0. GLOBAL STYLE
# ------------------------------------------------------------
NEON_BLUE   = "#00E5FF"
NEON_PINK   = "#FF2CFD"
NEON_GREEN  = "#00FF85"
NEON_ORANGE = "#FF9F1C"
NEON_YELLOW = "#FFE600"
NEON_PURPLE = "#9D4EDD"
NEON_CYAN2  = "#6EF7FF"
GRID_COLOR  = "rgba(255,255,255,0.08)"
BG_COLOR    = "#05070D"
PAPER_COLOR = "#05070D"
TEXT_COLOR  = "#E8F1FF"

def neon_layout(fig, title, height=750, width=1300):
    """Apply a dark neon style to a Plotly figure."""
    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(size=24, color=TEXT_COLOR)
        ),
        paper_bgcolor=PAPER_COLOR,
        plot_bgcolor=BG_COLOR,
        font=dict(color=TEXT_COLOR, size=13),
        height=height,
        width=width,
        legend=dict(
            bgcolor="rgba(0,0,0,0.0)",
            bordercolor="rgba(255,255,255,0.12)",
            borderwidth=1
        ),
        margin=dict(l=60, r=40, t=80, b=60)
    )
    return fig

def style_axes(fig, row=None, col=None, x_title=None, y_title=None):
    """Apply consistent axis style."""
    axis_kwargs = dict(
        showgrid=True,
        gridcolor=GRID_COLOR,
        zeroline=False,
        showline=True,
        linecolor="rgba(255,255,255,0.20)",
        ticks="outside",
        tickcolor="rgba(255,255,255,0.20)"
    )
    fig.update_xaxes(title_text=x_title, row=row, col=col, **axis_kwargs)
    fig.update_yaxes(title_text=y_title, row=row, col=col, **axis_kwargs)

# ------------------------------------------------------------
# 1. BASIC UTILITIES
# ------------------------------------------------------------
def sigmoid(z):
    """Numerically stable sigmoid."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))

def log_loss_binary(y_true, y_prob, eps=1e-9):
    """Binary logistic loss."""
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

def accuracy_binary(y_true, y_prob):
    """Binary classification accuracy."""
    return np.mean((y_prob >= 0.5).astype(int) == y_true)

def add_bias(X):
    """Append a bias column."""
    return np.hstack([X, np.ones((X.shape[0], 1))])

def make_dataset(seed=42, n_samples=500, noise=0.23):
    """
    Generate a simple nonlinear binary dataset.
    The goal is structural illustration, not benchmark optimization.
    """
    X, y = make_moons(n_samples=n_samples, noise=noise, random_state=seed)
    X = StandardScaler().fit_transform(X)
    return X, y.astype(float)

def normalize01(arr):
    """Min-max normalization for visualization only."""
    arr = np.asarray(arr, dtype=float)
    if len(arr) == 0:
        return arr
    span = arr.max() - arr.min()
    if span < 1e-12:
        return np.zeros_like(arr)
    return (arr - arr.min()) / span

def smooth_signal(signal, window=11):
    """Centered moving average with edge padding."""
    signal = np.asarray(signal, dtype=float)
    if len(signal) == 0 or window <= 1:
        return signal.copy()

    if window % 2 == 0:
        window += 1

    pad = window // 2
    padded = np.pad(signal, (pad, pad), mode="edge")
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(padded, kernel, mode="valid")

def robust_zscore(x):
    """Robust standardization via median and MAD."""
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return x.copy()

    med = np.median(x)
    mad = np.median(np.abs(x - med))
    if mad < 1e-12:
        return np.zeros_like(x)

    return (x - med) / (1.4826 * mad)

# ------------------------------------------------------------
# 2. GLOBAL GEOMETRIC DESCRIPTORS
# ------------------------------------------------------------
def total_variation(path):
    """
    Global first-order variation:
        V(path) = sum_t ||y_{t+1} - y_t||
    """
    path = np.asarray(path, dtype=float)
    if len(path) < 2:
        return 0.0
    diffs = np.diff(path, axis=0)
    return float(np.sum(np.linalg.norm(diffs, axis=1)))

def second_order_variation(path):
    """
    Global second-order variation:
        C(path) = sum_t ||y_{t+1} - 2 y_t + y_{t-1}||
    """
    path = np.asarray(path, dtype=float)
    if len(path) < 3:
        return 0.0
    sec = path[2:] - 2 * path[1:-1] + path[:-2]
    return float(np.sum(np.linalg.norm(sec, axis=1)))

def local_first_order(path):
    """Local first-order variation."""
    path = np.asarray(path, dtype=float)
    if len(path) < 2:
        return np.array([])
    diffs = np.diff(path, axis=0)
    return np.linalg.norm(diffs, axis=1)

def local_second_order(path):
    """Local second-order variation."""
    path = np.asarray(path, dtype=float)
    if len(path) < 3:
        return np.array([])
    sec = path[2:] - 2 * path[1:-1] + path[:-2]
    return np.linalg.norm(sec, axis=1)

# ------------------------------------------------------------
# 3. LOCAL JETS AND ADMISSIBLE OBSERVABLES
# ------------------------------------------------------------
def valid_centers(n, radius):
    """All valid centers for which a discrete jet is defined."""
    return np.arange(radius, n - radius)

def discrete_window(path, center, radius):
    """
    Extract a centered discrete window.
    This is the computational analogue of a local jet.
    """
    return np.asarray(path[center - radius:center + radius + 1], dtype=float)

def jet_oscillation(window):
    """
    Oscillation observable:
        sup_{i,j} ||w_i - w_j||
    """
    m = len(window)
    if m <= 1:
        return 0.0

    best = 0.0
    for i in range(m):
        for j in range(i + 1, m):
            val = np.linalg.norm(window[i] - window[j])
            if val > best:
                best = val
    return float(best)

def jet_triangular_excess(window):
    """
    Triangular excess observable:
        sup_{i<j<k} (d(i,j) + d(j,k) - d(i,k))
    """
    m = len(window)
    if m < 3:
        return 0.0

    best = 0.0
    for i in range(m - 2):
        for j in range(i + 1, m - 1):
            for k in range(j + 1, m):
                dij = np.linalg.norm(window[i] - window[j])
                djk = np.linalg.norm(window[j] - window[k])
                dik = np.linalg.norm(window[i] - window[k])
                val = dij + djk - dik
                if val > best:
                    best = val
    return float(best)

def jet_second_order_mass(window):
    """
    Second-order mass inside a window:
        sum_i ||w_{i+1} - 2 w_i + w_{i-1}||
    """
    window = np.asarray(window, dtype=float)
    if len(window) < 3:
        return 0.0

    sec = window[2:] - 2 * window[1:-1] + window[:-2]
    return float(np.sum(np.linalg.norm(sec, axis=1)))

def local_profile_from_observable(path, radius, observable_fn):
    """
    Build a structural profile from a local observable.
    Outside the admissible center set, the profile is zero.
    """
    path = np.asarray(path, dtype=float)
    n = len(path)
    profile = np.zeros(n, dtype=float)

    if n < 2 * radius + 1:
        return profile

    for t in valid_centers(n, radius):
        w = discrete_window(path, t, radius)
        profile[t] = observable_fn(w)

    return profile

# ------------------------------------------------------------
# 4. STRUCTURAL PROFILES, MASSES, SUPERLEVELS, BOUNDARIES
# ------------------------------------------------------------
def structural_mass(profile):
    """Total structural mass."""
    return float(np.sum(profile))

def superlevel_mask(profile, alpha):
    """Boolean superlevel set {t : profile(t) >= alpha}."""
    profile = np.asarray(profile, dtype=float)
    return profile >= alpha

def mask_to_intervals(mask):
    """Convert a boolean mask into contiguous intervals."""
    mask = np.asarray(mask, dtype=bool)
    if len(mask) == 0:
        return []

    intervals = []
    in_block = False
    start = None

    for t, flag in enumerate(mask):
        if flag and not in_block:
            start = t
            in_block = True
        elif not flag and in_block:
            intervals.append((start, t - 1))
            in_block = False

    if in_block:
        intervals.append((start, len(mask) - 1))

    return intervals

def boundary_points_from_mask(mask):
    """Extract discrete boundary points of a superlevel mask."""
    mask = np.asarray(mask, dtype=bool)
    if len(mask) == 0:
        return np.array([], dtype=int)

    pts = []
    if mask[0]:
        pts.append(0)

    for i in range(1, len(mask)):
        if mask[i] != mask[i - 1]:
            pts.append(i)

    if mask[-1]:
        pts.append(len(mask) - 1)

    return np.array(sorted(set(pts)), dtype=int)

# ------------------------------------------------------------
# 5. LOW-LEVEL INTERVAL UTILITIES
# ------------------------------------------------------------
def merge_close_intervals(intervals, gap=6):
    """Merge intervals separated by a short gap."""
    if len(intervals) <= 1:
        return intervals.copy()

    merged = [intervals[0]]
    for a, b in intervals[1:]:
        pa, pb = merged[-1]
        if a - pb - 1 <= gap:
            merged[-1] = (pa, b)
        else:
            merged.append((a, b))
    return merged

def filter_short_intervals(intervals, min_length=8):
    """Keep intervals with sufficient persistence."""
    return [(a, b) for (a, b) in intervals if (b - a + 1) >= min_length]

def mean_interval_length(intervals):
    """Mean interval length."""
    if len(intervals) == 0:
        return 0.0
    return float(np.mean([(b - a + 1) for (a, b) in intervals]))

def keep_top_intervals_by_mass(intervals, signal, max_intervals):
    """Keep the intervals with largest signal mass."""
    if max_intervals is None or len(intervals) <= max_intervals:
        return intervals

    scored = []
    for a, b in intervals:
        scored.append(((a, b), float(np.sum(signal[a:b + 1]))))

    scored = sorted(scored, key=lambda x: x[1], reverse=True)[:max_intervals]
    out = sorted([x[0] for x in scored], key=lambda z: z[0])
    return out

# ------------------------------------------------------------
# 6. THEORY-ALIGNED ENERGY
# ------------------------------------------------------------
def build_structural_energy(profile_param, profile_func, smooth_window=11, alpha=1.0, beta=1.0):
    """
    Combined structural energy built from mesoscopic profiles.
    """
    pp = smooth_signal(profile_param, smooth_window)
    pf = smooth_signal(profile_func, smooth_window)

    ppz = robust_zscore(pp)
    pfz = robust_zscore(pf)

    energy = alpha * ppz + beta * pfz
    energy = np.maximum(energy, 0.0)
    return energy, pp, pf

def build_trend_and_residual(energy, trend_window=31):
    """Build a slow trend and a nonnegative residual energy."""
    trend_window = max(3, int(trend_window))
    if trend_window % 2 == 0:
        trend_window += 1

    trend = smooth_signal(energy, trend_window)
    residual = np.maximum(np.asarray(energy, dtype=float) - trend, 0.0)
    return trend, residual

# ------------------------------------------------------------
# 7. MULTISCALE BOUNDARY SCORE
# ------------------------------------------------------------
def local_contrast_score(signal, radius):
    """
    Left-right mean contrast:
        |mean_left - mean_right|
    """
    signal = np.asarray(signal, dtype=float)
    n = len(signal)
    out = np.zeros(n, dtype=float)

    radius = max(1, int(radius))
    if n < 2 * radius + 1:
        return out

    for t in range(radius, n - radius):
        left = np.mean(signal[t - radius:t])
        right = np.mean(signal[t + 1:t + radius + 1])
        out[t] = abs(right - left)

    return out

def second_difference_score(signal):
    """Absolute discrete second derivative."""
    signal = np.asarray(signal, dtype=float)
    out = np.zeros(len(signal), dtype=float)
    if len(signal) < 3:
        return out
    out[1:-1] = np.abs(signal[2:] - 2 * signal[1:-1] + signal[:-2])
    return out

def build_boundary_score(
    smooth_profile_param,
    smooth_profile_func,
    energy,
    residual,
    contrast_radius=8,
    w_param=1.0,
    w_func=1.25,
    w_residual=0.90,
    w_curv=0.35
):
    """
    Multiscale structural boundary score.
    """
    spp = robust_zscore(smooth_profile_param)
    spf = robust_zscore(smooth_profile_func)
    enr = robust_zscore(np.asarray(energy, dtype=float))
    res = robust_zscore(np.asarray(residual, dtype=float))

    c_param = local_contrast_score(spp, contrast_radius)
    c_func = local_contrast_score(spf, contrast_radius)
    c_res = local_contrast_score(res, contrast_radius)
    curv = second_difference_score(enr)

    score = (
        w_param * c_param +
        w_func * c_func +
        w_residual * c_res +
        w_curv * curv
    )
    score = np.maximum(score, 0.0)

    return {
        "boundary_score": score,
        "contrast_param": c_param,
        "contrast_func": c_func,
        "contrast_residual": c_res,
        "curvature_energy": curv
    }

# ------------------------------------------------------------
# 8. PEAK SELECTION AND EVENT PARSING
# ------------------------------------------------------------
def local_maxima_indices(signal):
    """All local maxima of a 1D signal."""
    signal = np.asarray(signal, dtype=float)
    n = len(signal)
    if n < 3:
        return np.array([], dtype=int)

    peaks = []
    for i in range(1, n - 1):
        if signal[i] >= signal[i - 1] and signal[i] >= signal[i + 1] and signal[i] > 0:
            peaks.append(i)

    return np.array(peaks, dtype=int)

def select_separated_peaks(score, peaks, min_separation=18, max_peaks=3):
    """Greedy peak selection with hard separation."""
    if len(peaks) == 0:
        return np.array([], dtype=int)

    peaks = sorted(list(peaks), key=lambda i: score[i], reverse=True)
    chosen = []

    for p in peaks:
        if all(abs(p - q) >= min_separation for q in chosen):
            chosen.append(p)
        if len(chosen) >= max_peaks:
            break

    return np.array(sorted(chosen), dtype=int)

def expand_peak_to_interval(score, peak, medium_threshold, max_half_width=16):
    """
    Expand a peak into a transition interval while the score stays above
    a medium threshold.
    """
    score = np.asarray(score, dtype=float)
    n = len(score)

    left = int(peak)
    right = int(peak)

    while left > 0 and (peak - left) < max_half_width and score[left - 1] >= medium_threshold:
        left -= 1

    while right < n - 1 and (right - peak) < max_half_width and score[right + 1] >= medium_threshold:
        right += 1

    return (left, right)

def boundary_events_from_score(
    boundary_score,
    q_peak=0.91,
    q_medium=0.64,
    min_separation=20,
    max_events=3,
    burn_in_exclusion=0.10,
    max_half_width=16,
    min_event_length=6
):
    """
    Parse events from a multiscale boundary score.
    """
    score = np.asarray(boundary_score, dtype=float)
    n = len(score)

    if n == 0 or np.max(score) <= 1e-12:
        return {
            "boundary_peaks": np.array([], dtype=int),
            "event_intervals": [],
            "transition_intervals": []
        }

    start_idx = int(np.floor(burn_in_exclusion * n))
    start_idx = min(max(0, start_idx), n - 1)

    local = score[start_idx:]
    if len(local) == 0 or np.max(local) <= 1e-12:
        return {
            "boundary_peaks": np.array([], dtype=int),
            "event_intervals": [],
            "transition_intervals": []
        }

    all_peaks = local_maxima_indices(local) + start_idx
    if len(all_peaks) == 0:
        return {
            "boundary_peaks": np.array([], dtype=int),
            "event_intervals": [],
            "transition_intervals": []
        }

    peak_thr = np.quantile(local, q_peak)
    medium_thr = np.quantile(local, q_medium)

    strong_peaks = np.array([p for p in all_peaks if score[p] >= peak_thr], dtype=int)
    if len(strong_peaks) == 0:
        strong_peaks = all_peaks

    selected_peaks = select_separated_peaks(
        score=score,
        peaks=strong_peaks,
        min_separation=min_separation,
        max_peaks=max_events
    )

    event_intervals = [
        expand_peak_to_interval(score, p, medium_threshold=medium_thr, max_half_width=max_half_width)
        for p in selected_peaks
    ]

    event_intervals = merge_close_intervals(sorted(event_intervals, key=lambda z: z[0]), gap=4)
    event_intervals = filter_short_intervals(event_intervals, min_length=min_event_length)
    event_intervals = keep_top_intervals_by_mass(event_intervals, score, max_events)

    transition_thr = np.quantile(local, max(0.45, q_medium - 0.08))
    transition_intervals = [
        expand_peak_to_interval(score, p, medium_threshold=transition_thr, max_half_width=max_half_width + 6)
        for p in selected_peaks
    ]
    transition_intervals = merge_close_intervals(sorted(transition_intervals, key=lambda z: z[0]), gap=4)
    transition_intervals = filter_short_intervals(transition_intervals, min_length=min_event_length)

    return {
        "boundary_peaks": selected_peaks,
        "event_intervals": event_intervals,
        "transition_intervals": transition_intervals
    }

def event_intervals_to_macro_regimes(n_steps, event_intervals):
    """Macro-regimes are the complementary intervals between event blocks."""
    if n_steps <= 0:
        return []

    if len(event_intervals) == 0:
        return [{
            "start": 0,
            "end": n_steps - 1,
            "length": n_steps,
            "signature": ("macro_regime", 0)
        }]

    regimes = []
    current = 0
    rid = 0

    for a, b in event_intervals:
        if current <= a - 1:
            regimes.append({
                "start": current,
                "end": a - 1,
                "length": a - current,
                "signature": ("macro_regime", rid)
            })
            rid += 1
        current = b + 1

    if current <= n_steps - 1:
        regimes.append({
            "start": current,
            "end": n_steps - 1,
            "length": n_steps - current,
            "signature": ("macro_regime", rid)
        })

    return regimes

def regime_word(regimes):
    """Word-like representation of the macro-regime decomposition."""
    return [(r["signature"], int(r["length"])) for r in regimes]

def number_of_regime_changes(regimes):
    """Number of regime changes."""
    return max(0, len(regimes) - 1)

# ------------------------------------------------------------
# 9. STRUCTURAL INDICES
# ------------------------------------------------------------
def peak_concentration_index(profile, top_fraction=0.10):
    """
    Fraction of mass concentrated in the top local sites.
    Must lie in [0, 1].
    """
    profile = np.asarray(profile, dtype=float)
    profile = np.clip(profile, 0.0, None)

    if len(profile) == 0:
        return 0.0

    total = float(np.sum(profile))
    if total <= 1e-12:
        return 0.0

    k = max(1, int(np.ceil(top_fraction * len(profile))))
    topk = np.sort(profile)[-k:]
    value = float(np.sum(topk) / total)
    return min(max(value, 0.0), 1.0)

def boundary_localization_index(profile, boundaries, radius=5):
    """
    Fraction of structural mass near known schedule boundaries.
    Must lie in [0, 1].
    """
    profile = np.asarray(profile, dtype=float)
    profile = np.clip(profile, 0.0, None)

    if len(profile) == 0:
        return 0.0

    total = float(np.sum(profile))
    if total <= 1e-12:
        return 0.0

    if boundaries is None or len(boundaries) == 0:
        return 0.0

    mask = np.zeros(len(profile), dtype=bool)
    for b in boundaries:
        left = max(0, int(b) - radius)
        right = min(len(profile), int(b) + radius + 1)
        mask[left:right] = True

    value = float(np.sum(profile[mask]) / total)
    return min(max(value, 0.0), 1.0)

def top_mass_localization_index(score, boundaries, radius=8, top_fraction=0.12):
    """
    Auxiliary diagnostic only.
    Fraction of the top structural score mass concentrated near known boundaries.
    """
    score = np.asarray(score, dtype=float)
    score = np.clip(score, 0.0, None)

    if len(score) == 0 or np.sum(score) <= 1e-12:
        return 0.0

    if boundaries is None or len(boundaries) == 0:
        return 0.0

    k = max(1, int(np.ceil(top_fraction * len(score))))
    thr = np.partition(score, -k)[-k]

    top_score = np.where(score >= thr, score, 0.0)
    total = float(np.sum(top_score))
    if total <= 1e-12:
        return 0.0

    mask = np.zeros(len(score), dtype=bool)
    for b in boundaries:
        left = max(0, int(b) - radius)
        right = min(len(score), int(b) + radius + 1)
        mask[left:right] = True

    value = float(np.sum(top_score[mask]) / total)
    return min(max(value, 0.0), 1.0)

def boundary_peak_recall(boundary_peaks, boundaries, tol=8):
    """
    Fraction of known boundaries that are matched by at least one detected peak.
    """
    peaks = np.asarray(boundary_peaks, dtype=int)

    if boundaries is None or len(boundaries) == 0:
        return 0.0
    if len(peaks) == 0:
        return 0.0

    hits = 0
    for b in boundaries:
        if np.any(np.abs(peaks - int(b)) <= tol):
            hits += 1

    return hits / len(boundaries)

def boundary_peak_precision(boundary_peaks, boundaries, tol=8):
    """
    Fraction of detected peaks that lie near a known boundary.
    """
    peaks = np.asarray(boundary_peaks, dtype=int)

    if len(peaks) == 0:
        return 0.0
    if boundaries is None or len(boundaries) == 0:
        return 0.0

    good = 0
    for p in peaks:
        if any(abs(int(p) - int(b)) <= tol for b in boundaries):
            good += 1

    return good / len(peaks)

def normalized_max_boundary_score(score):
    """
    Auxiliary diagnostic only.
    Dimensionless version of the boundary-score maximum.
    """
    score = np.asarray(score, dtype=float)
    if len(score) == 0:
        return 0.0

    z = robust_zscore(score)
    return float(np.max(z)) if len(z) else 0.0

def approximate_budget_ratio(profile, alpha, lam):
    """
    Computational counterpart of the structural budget:
        M / (alpha * lambda)
    """
    denom = alpha * lam
    if denom <= 1e-12:
        return np.nan
    return float(np.sum(profile) / denom)

# ------------------------------------------------------------
# 10. TRAINING LOOP
# ------------------------------------------------------------
def train_logistic_with_schedule(X, y, lr_schedule, init_seed=0, tag="run"):
    """Train a small logistic model and record trajectories."""
    rng = np.random.default_rng(init_seed)

    Xb = add_bias(X)
    n, d = Xb.shape
    w = 0.05 * rng.standard_normal(d)

    anchor_Xb = Xb.copy()

    param_path = []
    func_path = []
    losses = []
    accs = []
    grad_norms = []
    lrs = []

    for lr in lr_schedule:
        logits = Xb @ w
        probs = sigmoid(logits)
        grad = (Xb.T @ (probs - y)) / n

        param_path.append(w.copy())
        func_path.append(sigmoid(anchor_Xb @ w))
        losses.append(log_loss_binary(y, probs))
        accs.append(accuracy_binary(y, probs))
        grad_norms.append(np.linalg.norm(grad))
        lrs.append(lr)

        w = w - lr * grad

    logits = Xb @ w
    probs = sigmoid(logits)
    grad = (Xb.T @ (probs - y)) / n

    param_path.append(w.copy())
    func_path.append(sigmoid(anchor_Xb @ w))
    losses.append(log_loss_binary(y, probs))
    accs.append(accuracy_binary(y, probs))
    grad_norms.append(np.linalg.norm(grad))
    lrs.append(lrs[-1] if len(lrs) > 0 else 0.0)

    return {
        "tag": tag,
        "X": X,
        "y": y,
        "param_path": np.array(param_path),
        "func_path": np.array(func_path),
        "losses": np.array(losses),
        "accs": np.array(accs),
        "grad_norms": np.array(grad_norms),
        "lrs": np.array(lrs)
    }

# ------------------------------------------------------------
# 11. THEORY-ALIGNED EXTRACTION PIPELINE
# ------------------------------------------------------------
def extract_structural_descriptors(
    run,
    phase_boundaries=None,
    jet_radius=6,
    smooth_window=11,
    alpha_quantile=0.85,
    lambda_min=8,
    osc_weight=0.20,
    exc_weight=1.00,
    som_weight=1.00,
    trend_window=31,
    contrast_radius=8,
    q_boundary_peak=0.91,
    q_boundary_medium=0.64,
    min_peak_separation=20,
    max_events=3,
    burn_in_exclusion=0.10,
    max_half_width=16,
    min_event_length=6
):
    """
    Main structural extractor.
    """
    P = np.asarray(run["param_path"], dtype=float)
    F = np.asarray(run["func_path"], dtype=float)

    # ---- Global geometry
    param_V = total_variation(P)
    param_C = second_order_variation(P)
    func_V = total_variation(F)
    func_C = second_order_variation(F)

    local_V_param = local_first_order(P)
    local_C_param = local_second_order(P)
    local_V_func = local_first_order(F)
    local_C_func = local_second_order(F)

    # ---- Local observable profiles
    prof_osc_param = local_profile_from_observable(P, jet_radius, jet_oscillation)
    prof_osc_func  = local_profile_from_observable(F, jet_radius, jet_oscillation)

    prof_exc_param = local_profile_from_observable(P, jet_radius, jet_triangular_excess)
    prof_exc_func  = local_profile_from_observable(F, jet_radius, jet_triangular_excess)

    prof_som_param = local_profile_from_observable(P, jet_radius, jet_second_order_mass)
    prof_som_func  = local_profile_from_observable(F, jet_radius, jet_second_order_mass)

    # ---- Canonical structural profiles
    profile_param = (
        osc_weight * prof_osc_param +
        exc_weight * prof_exc_param +
        som_weight * prof_som_param
    )

    profile_func = (
        osc_weight * prof_osc_func +
        exc_weight * prof_exc_func +
        som_weight * prof_som_func
    )

    mass_param = structural_mass(profile_param)
    mass_func = structural_mass(profile_func)

    # ---- Theory-aligned structural energy
    energy, smooth_param, smooth_func = build_structural_energy(
        profile_param,
        profile_func,
        smooth_window=smooth_window,
        alpha=1.0,
        beta=1.0
    )

    # ---- Original superlevel geometry stays attached to original energy
    alpha = float(np.quantile(energy, alpha_quantile)) if len(energy) else 0.0
    super_mask = superlevel_mask(energy, alpha)
    super_intervals = mask_to_intervals(super_mask)
    super_boundaries = boundary_points_from_mask(super_mask)

    # ---- Trend / residual
    trend_energy, residual_energy = build_trend_and_residual(
        energy,
        trend_window=trend_window
    )

    # ---- Corrected multiscale boundary score
    bscore_parts = build_boundary_score(
        smooth_profile_param=smooth_param,
        smooth_profile_func=smooth_func,
        energy=energy,
        residual=residual_energy,
        contrast_radius=contrast_radius,
        w_param=1.00,
        w_func=1.25,
        w_residual=0.90,
        w_curv=0.35
    )

    boundary_score = bscore_parts["boundary_score"]
    contrast_param = bscore_parts["contrast_param"]
    contrast_func = bscore_parts["contrast_func"]
    contrast_residual = bscore_parts["contrast_residual"]
    curvature_energy = bscore_parts["curvature_energy"]

    # ---- Corrected event parsing
    parsed = boundary_events_from_score(
        boundary_score=boundary_score,
        q_peak=q_boundary_peak,
        q_medium=q_boundary_medium,
        min_separation=min_peak_separation,
        max_events=max_events,
        burn_in_exclusion=burn_in_exclusion,
        max_half_width=max_half_width,
        min_event_length=min_event_length
    )

    boundary_peaks = parsed["boundary_peaks"]
    event_intervals = parsed["event_intervals"]
    transition_intervals = parsed["transition_intervals"]

    regimes = event_intervals_to_macro_regimes(len(boundary_score), event_intervals)
    word = regime_word(regimes)

    # ---- Main evaluation indices
    peak_param = peak_concentration_index(profile_param)
    peak_func = peak_concentration_index(profile_func)

    loc_param = boundary_localization_index(profile_param, phase_boundaries)
    loc_func = boundary_localization_index(profile_func, phase_boundaries)

    boundary_peak_recall_score = boundary_peak_recall(
        boundary_peaks,
        phase_boundaries,
        tol=8
    )

    boundary_peak_precision_score = boundary_peak_precision(
        boundary_peaks,
        phase_boundaries,
        tol=8
    )

    # ---- Auxiliary diagnostics only
    boundary_score_loc = top_mass_localization_index(
        boundary_score,
        phase_boundaries,
        radius=8,
        top_fraction=0.12
    )

    max_boundary_score_norm = normalized_max_boundary_score(boundary_score)

    # ---- Budget proxy
    budget_ratio = approximate_budget_ratio(energy, max(alpha, 1e-9), lambda_min)

    return {
        "tag": run["tag"],
        "final_loss": float(run["losses"][-1]),
        "final_acc": float(run["accs"][-1]),

        # Global descriptors
        "param_V": param_V,
        "param_C": param_C,
        "func_V": func_V,
        "func_C": func_C,

        # Local baseline signals
        "local_V_param": local_V_param,
        "local_C_param": local_C_param,
        "local_V_func": local_V_func,
        "local_C_func": local_C_func,

        # Observable profiles
        "prof_osc_param": prof_osc_param,
        "prof_osc_func": prof_osc_func,
        "prof_exc_param": prof_exc_param,
        "prof_exc_func": prof_exc_func,
        "prof_som_param": prof_som_param,
        "prof_som_func": prof_som_func,

        # Canonical structural profiles
        "profile_param": profile_param,
        "profile_func": profile_func,
        "mass_param": mass_param,
        "mass_func": mass_func,

        # Energy layer
        "energy": energy,
        "smooth_profile_param": smooth_param,
        "smooth_profile_func": smooth_func,
        "alpha": alpha,
        "trend_energy": trend_energy,
        "residual_energy": residual_energy,

        # Superlevel geometry
        "superlevel_mask": super_mask,
        "superlevel_intervals": super_intervals,
        "superlevel_boundaries": super_boundaries,

        # Corrected boundary score
        "boundary_score": boundary_score,
        "contrast_param": contrast_param,
        "contrast_func": contrast_func,
        "contrast_residual": contrast_residual,
        "curvature_energy": curvature_energy,
        "boundary_peaks": boundary_peaks,

        # Events / transitions / regimes
        "event_intervals": event_intervals,
        "transition_intervals": transition_intervals,
        "regimes": regimes,
        "regime_word": word,

        # Main scalar summaries
        "peak_param": peak_param,
        "peak_func": peak_func,
        "boundary_loc_param": loc_param,
        "boundary_loc_func": loc_func,
        "boundary_peak_recall": boundary_peak_recall_score,
        "boundary_peak_precision": boundary_peak_precision_score,
        "n_events": len(event_intervals),
        "n_boundaries": len(boundary_peaks),
        "n_regimes": len(regimes),
        "n_regime_changes": number_of_regime_changes(regimes),
        "n_transition_zones": len(transition_intervals),
        "mean_transition_length": mean_interval_length(transition_intervals),
        "n_superlevel_components": len(super_intervals),
        "n_superlevel_boundaries": len(super_boundaries),
        "budget_ratio": budget_ratio,
        "max_energy": float(np.max(energy)) if len(energy) else 0.0,
        "max_residual_energy": float(np.max(residual_energy)) if len(residual_energy) else 0.0,
        "max_profile_param": float(np.max(profile_param)) if len(profile_param) else 0.0,
        "max_profile_func": float(np.max(profile_func)) if len(profile_func) else 0.0,

        # Auxiliary diagnostics (kept internal / optional)
        "boundary_score_loc": boundary_score_loc,
        "max_boundary_score": float(np.max(boundary_score)) if len(boundary_score) else 0.0,
        "max_boundary_score_norm": max_boundary_score_norm
    }

# ------------------------------------------------------------
# 12. SCHEDULE FAMILIES
# ------------------------------------------------------------
T = 180

def homogeneous_schedule(T):
    return np.full(T, 0.18)

def regime_change_schedule(T):
    return np.r_[np.full(T // 2, 0.08), np.full(T - T // 2, 0.48)]

def three_phase_schedule(T):
    return np.r_[np.full(45, 0.52), np.full(70, 0.18), np.full(T - 115, 0.04)]

def pulse_schedule(T):
    base = np.full(T, 0.18)
    base[70:90] = 0.55
    return base

def oscillatory_schedule(T):
    x = np.linspace(0, 4 * np.pi, T)
    return 0.18 + 0.08 * (1.0 + np.sin(x))

# ------------------------------------------------------------
# 13. MULTI-SEED RUNNER
# ------------------------------------------------------------
def run_family(
    tag,
    schedule_fn,
    seeds,
    data_seed_base=1000,
    phase_boundaries=None,
    **extract_kwargs
):
    runs = []
    descs = []

    for seed in seeds:
        X, y = make_dataset(seed=data_seed_base + seed)
        schedule = schedule_fn(T)

        run = train_logistic_with_schedule(X, y, schedule, init_seed=seed, tag=tag)
        desc = extract_structural_descriptors(
            run,
            phase_boundaries=phase_boundaries,
            **extract_kwargs
        )
        runs.append(run)
        descs.append(desc)

    df = pd.DataFrame([
        {k: v for k, v in d.items() if not isinstance(v, (np.ndarray, list, dict))}
        for d in descs
    ])
    return runs, descs, df

SEEDS = list(range(20))

DEFAULT_JET_RADIUS = 6
DEFAULT_SMOOTH_WINDOW = 11
DEFAULT_ALPHA_QUANTILE = 0.85
DEFAULT_LAMBDA_MIN = 8
DEFAULT_OSC_WEIGHT = 0.20
DEFAULT_EXC_WEIGHT = 1.00
DEFAULT_SOM_WEIGHT = 1.00

# Boundary parser defaults
DEFAULT_TREND_WINDOW = 31
DEFAULT_CONTRAST_RADIUS = 8
DEFAULT_Q_BOUNDARY_PEAK = 0.91
DEFAULT_Q_BOUNDARY_MEDIUM = 0.64
DEFAULT_MIN_PEAK_SEPARATION = 20
DEFAULT_MAX_EVENTS = 3
DEFAULT_BURN_IN_EXCLUSION = 0.10
DEFAULT_MAX_HALF_WIDTH = 16
DEFAULT_MIN_EVENT_LENGTH = 6

COMMON_EXTRACT_KWARGS = dict(
    jet_radius=DEFAULT_JET_RADIUS,
    smooth_window=DEFAULT_SMOOTH_WINDOW,
    alpha_quantile=DEFAULT_ALPHA_QUANTILE,
    lambda_min=DEFAULT_LAMBDA_MIN,
    osc_weight=DEFAULT_OSC_WEIGHT,
    exc_weight=DEFAULT_EXC_WEIGHT,
    som_weight=DEFAULT_SOM_WEIGHT,
    trend_window=DEFAULT_TREND_WINDOW,
    contrast_radius=DEFAULT_CONTRAST_RADIUS,
    q_boundary_peak=DEFAULT_Q_BOUNDARY_PEAK,
    q_boundary_medium=DEFAULT_Q_BOUNDARY_MEDIUM,
    min_peak_separation=DEFAULT_MIN_PEAK_SEPARATION,
    max_events=DEFAULT_MAX_EVENTS,
    burn_in_exclusion=DEFAULT_BURN_IN_EXCLUSION,
    max_half_width=DEFAULT_MAX_HALF_WIDTH,
    min_event_length=DEFAULT_MIN_EVENT_LENGTH
)

runs_h, desc_h, df_h = run_family(
    "Homogeneous dynamics",
    homogeneous_schedule,
    SEEDS,
    data_seed_base=1000,
    phase_boundaries=[],
    **COMMON_EXTRACT_KWARGS
)

runs_r, desc_r, df_r = run_family(
    "Regime-change dynamics",
    regime_change_schedule,
    SEEDS,
    data_seed_base=2000,
    phase_boundaries=[T // 2],
    **COMMON_EXTRACT_KWARGS
)

runs_3, desc_3, df_3 = run_family(
    "Three-phase dynamics",
    three_phase_schedule,
    SEEDS,
    data_seed_base=3000,
    phase_boundaries=[45, 115],
    **COMMON_EXTRACT_KWARGS
)

runs_p, desc_p, df_p = run_family(
    "Pulse dynamics",
    pulse_schedule,
    SEEDS,
    data_seed_base=4000,
    phase_boundaries=[70, 90],
    **COMMON_EXTRACT_KWARGS
)

runs_o, desc_o, df_o = run_family(
    "Oscillatory dynamics",
    oscillatory_schedule,
    SEEDS,
    data_seed_base=5000,
    phase_boundaries=[],
    **COMMON_EXTRACT_KWARGS
)

summary_df = pd.concat([df_h, df_r, df_3, df_p, df_o], ignore_index=True)

# ------------------------------------------------------------
# 14. AUDIT
# ------------------------------------------------------------
def audit_bounded_columns(df, columns):
    """Verify that bounded structural indices lie in [0, 1]."""
    print("\n=== AUDIT OF BOUNDED STRUCTURAL INDICES ===")
    ok = True
    for col in columns:
        cmin = float(df[col].min())
        cmax = float(df[col].max())
        print(f"{col}: min={cmin:.6f}, max={cmax:.6f}")
        if cmin < -1e-9 or cmax > 1 + 1e-9:
            ok = False
    if not ok:
        raise ValueError("At least one bounded structural index is outside [0, 1].")
    print("All bounded structural indices lie in [0, 1].")

audit_bounded_columns(
    summary_df,
    [
        "peak_param",
        "peak_func",
        "boundary_loc_param",
        "boundary_loc_func",
        "boundary_peak_recall",
        "boundary_peak_precision"
    ]
)

# ------------------------------------------------------------
# 15. PAPER TABLES
# ------------------------------------------------------------
def mean_std_table(df, metrics, decimals=6):
    """Grouped mean/std table."""
    return df.groupby("tag")[metrics].agg(["mean", "std"]).round(decimals)

table_A_metrics = [
    "param_V", "func_V", "param_C", "func_C",
    "final_loss", "final_acc"
]

table_B_metrics = [
    "mass_param", "mass_func",
    "peak_param", "peak_func",
    "boundary_loc_param", "boundary_loc_func",
    "boundary_peak_recall",
    "boundary_peak_precision",
    "max_energy",
    "max_residual_energy",
    "max_profile_param", "max_profile_func"
]

table_C_metrics = [
    "n_events",
    "n_boundaries",
    "n_regimes",
    "n_regime_changes",
    "n_transition_zones",
    "mean_transition_length",
    "n_superlevel_components",
    "n_superlevel_boundaries",
    "budget_ratio"
]

paper_table_A = mean_std_table(summary_df, table_A_metrics)
paper_table_B = mean_std_table(summary_df, table_B_metrics)
paper_table_C = mean_std_table(summary_df, table_C_metrics)

print("\n=== PAPER TABLE A: GLOBAL GEOMETRIC SUMMARY ===")
print(paper_table_A)

print("\n=== PAPER TABLE B: STRUCTURAL PROFILE SUMMARY ===")
print(paper_table_B)

print("\n=== PAPER TABLE C: EVENTS / REGIMES / SUPERLEVELS ===")
print(paper_table_C)

# ------------------------------------------------------------
# 16. AGGREGATION UTILITIES FOR FIGURES
# ------------------------------------------------------------
def aggregate_curves(desc_list, key):
    """Aggregate curves with NaN padding. Returns mean and std."""
    curves = [np.asarray(d[key], dtype=float) for d in desc_list]
    max_len = max(len(c) for c in curves)
    M = np.full((len(curves), max_len), np.nan)

    for i, c in enumerate(curves):
        M[i, :len(c)] = c

    return np.nanmean(M, axis=0), np.nanstd(M, axis=0)

def add_mean_std_band(fig, x, mean, std, name, color, row, col,
                      legendgroup=None, showlegend=True):
    """Add a mean curve with one-standard-deviation band."""
    upper = mean + std
    lower = np.maximum(mean - std, 0)
    lg = legendgroup or name

    fig.add_trace(go.Scatter(
        x=x, y=upper,
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        legendgroup=lg,
        hoverinfo="skip"
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=x, y=lower,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor="rgba(255,255,255,0.08)",
        showlegend=False,
        legendgroup=lg,
        hoverinfo="skip"
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=x, y=mean,
        mode="lines",
        line=dict(color=color, width=3),
        name=name,
        legendgroup=lg,
        showlegend=showlegend
    ), row=row, col=col)

def select_representative_run(descs, key="mass_func"):
    """Pick the run closest to the median for a scalar descriptor."""
    vals = np.array([d[key] for d in descs], dtype=float)
    target = np.median(vals)
    return int(np.argmin(np.abs(vals - target)))

# ------------------------------------------------------------
# 17. FIGURE 1 — GLOBAL BOX PLOTS
# ------------------------------------------------------------
def plot_global_boxplots(df):
    """Figure 1: global first/second-order geometry."""
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Parametric first-order variation V",
            "Functional first-order variation V",
            "Parametric second-order variation C",
            "Functional second-order variation C"
        )
    )

    order = [
        "Homogeneous dynamics",
        "Regime-change dynamics",
        "Three-phase dynamics",
        "Pulse dynamics",
        "Oscillatory dynamics"
    ]

    colors = {
        "Homogeneous dynamics": NEON_BLUE,
        "Regime-change dynamics": NEON_PINK,
        "Three-phase dynamics": NEON_ORANGE,
        "Pulse dynamics": NEON_GREEN,
        "Oscillatory dynamics": NEON_PURPLE
    }

    for tag in order:
        sub = df[df["tag"] == tag]

        fig.add_trace(go.Box(
            y=sub["param_V"], name=tag, marker_color=colors[tag],
            boxmean=True, showlegend=False
        ), row=1, col=1)

        fig.add_trace(go.Box(
            y=sub["func_V"], name=tag, marker_color=colors[tag],
            boxmean=True, showlegend=False
        ), row=1, col=2)

        fig.add_trace(go.Box(
            y=sub["param_C"], name=tag, marker_color=colors[tag],
            boxmean=True, showlegend=False
        ), row=2, col=1)

        fig.add_trace(go.Box(
            y=sub["func_C"], name=tag, marker_color=colors[tag],
            boxmean=True, showlegend=False
        ), row=2, col=2)

    neon_layout(
        fig,
        "Figure 1 — Global Geometric Descriptors Across Run Families",
        height=920,
        width=1400
    )

    fig.update_layout(
        margin=dict(l=60, r=40, t=130, b=80),
        title=dict(
            text="Figure 1 — Global Geometric Descriptors Across Run Families",
            x=0.5,
            xanchor="center",
            y=0.98
        )
    )

    style_axes(fig, 1, 1, None, "Magnitude")
    style_axes(fig, 1, 2, None, "Magnitude")
    style_axes(fig, 2, 1, None, "Magnitude")
    style_axes(fig, 2, 2, None, "Magnitude")

    for r, c in [(1, 1), (1, 2), (2, 1), (2, 2)]:
        fig.update_xaxes(
            categoryorder="array",
            categoryarray=order,
            tickangle=-15,
            row=r,
            col=c
        )

    return fig

fig1 = plot_global_boxplots(summary_df)
fig1.show()

# ------------------------------------------------------------
# 18. FIGURE 2 — STRUCTURAL PROFILES
# ------------------------------------------------------------
def plot_structural_profiles(desc_dict):
    """Figure 2: mean structural profiles across families."""
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "Structural profile in parameter space",
            "Structural profile in function space"
        )
    )

    color_map = {
        "Homogeneous dynamics": NEON_BLUE,
        "Regime-change dynamics": NEON_PINK,
        "Three-phase dynamics": NEON_ORANGE,
        "Pulse dynamics": NEON_GREEN,
        "Oscillatory dynamics": NEON_PURPLE
    }

    order = [
        "Homogeneous dynamics",
        "Regime-change dynamics",
        "Three-phase dynamics",
        "Pulse dynamics",
        "Oscillatory dynamics"
    ]

    for tag in order:
        descs = desc_dict[tag]
        mp, sp = aggregate_curves(descs, "profile_param")
        mf, sf = aggregate_curves(descs, "profile_func")

        add_mean_std_band(
            fig, np.arange(len(mp)), mp, sp, tag, color_map[tag],
            1, 1, legendgroup=tag, showlegend=True
        )
        add_mean_std_band(
            fig, np.arange(len(mf)), mf, sf, tag, color_map[tag],
            1, 2, legendgroup=tag, showlegend=False
        )

    for col in [1, 2]:
        fig.add_vline(x=45, line=dict(color=NEON_GREEN, width=2, dash="dot"), row=1, col=col)
        fig.add_vline(x=90, line=dict(color=NEON_YELLOW, width=2, dash="dash"), row=1, col=col)
        fig.add_vline(x=115, line=dict(color=NEON_GREEN, width=2, dash="dot"), row=1, col=col)

    neon_layout(fig, "Figure 2 — Theory-Aligned Structural Profiles", height=680, width=1450)
    style_axes(fig, 1, 1, "Iteration", "Structural profile magnitude")
    style_axes(fig, 1, 2, "Iteration", "Structural profile magnitude")

    fig.update_layout(
        margin=dict(l=60, r=40, t=150, b=70),
        title=dict(
            text="Figure 2 — Theory-Aligned Structural Profiles",
            x=0.5,
            xanchor="center",
            y=0.98
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.08,
            xanchor="center",
            x=0.5,
            title_text="",
            bgcolor="rgba(0,0,0,0.0)",
            bordercolor="rgba(255,255,255,0.12)",
            borderwidth=1
        )
    )

    return fig

fig2 = plot_structural_profiles({
    "Homogeneous dynamics": desc_h,
    "Regime-change dynamics": desc_r,
    "Three-phase dynamics": desc_3,
    "Pulse dynamics": desc_p,
    "Oscillatory dynamics": desc_o
})
fig2.show()

# ------------------------------------------------------------
# 19. FIGURE 3 — REPRESENTATIVE RUN
# ------------------------------------------------------------
rep_three = select_representative_run(desc_3, key="mass_func")

def plot_representative_run(run, desc, title):
    """
    Figure 3: representative run in parameter space, function space,
    structural profiles, and corrected boundary parser.
    """
    P = np.asarray(run["param_path"], dtype=float)
    F = np.asarray(run["func_path"], dtype=float)

    pca = PCA(n_components=2)
    F2 = pca.fit_transform(F)

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Parameter trajectory",
            "Function-space trajectory (PCA projection)",
            "Structural profiles",
            "Boundary score and peaks"
        )
    )

    fig.add_trace(go.Scatter(
        x=P[:, 0], y=P[:, 1],
        mode="lines+markers",
        line=dict(color=NEON_BLUE, width=3),
        marker=dict(size=4, color=NEON_BLUE),
        name="Parameter path"
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=[P[0, 0]], y=[P[0, 1]],
        mode="markers",
        marker=dict(size=10, color=NEON_GREEN, symbol="diamond"),
        showlegend=False
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=[P[-1, 0]], y=[P[-1, 1]],
        mode="markers",
        marker=dict(size=10, color=NEON_YELLOW, symbol="x"),
        showlegend=False
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=F2[:, 0], y=F2[:, 1],
        mode="lines+markers",
        line=dict(color=NEON_PINK, width=3),
        marker=dict(size=4, color=NEON_PINK),
        name="Function path"
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=[F2[0, 0]], y=[F2[0, 1]],
        mode="markers",
        marker=dict(size=10, color=NEON_GREEN, symbol="diamond"),
        showlegend=False
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=[F2[-1, 0]], y=[F2[-1, 1]],
        mode="markers",
        marker=dict(size=10, color=NEON_YELLOW, symbol="x"),
        showlegend=False
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=np.arange(len(desc["profile_param"])),
        y=normalize01(desc["profile_param"]),
        mode="lines",
        line=dict(color=NEON_GREEN, width=3),
        name="Profile param"
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=np.arange(len(desc["profile_func"])),
        y=normalize01(desc["profile_func"]),
        mode="lines",
        line=dict(color=NEON_ORANGE, width=3),
        name="Profile func"
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=np.arange(len(desc["boundary_score"])),
        y=desc["boundary_score"],
        mode="lines",
        line=dict(color=NEON_PURPLE, width=3),
        name="Boundary score"
    ), row=2, col=2)

    peaks = np.asarray(desc["boundary_peaks"], dtype=int)
    if len(peaks) > 0:
        fig.add_trace(go.Scatter(
            x=peaks,
            y=desc["boundary_score"][peaks],
            mode="markers",
            marker=dict(size=10, color=NEON_YELLOW, symbol="diamond"),
            name="Boundary peaks"
        ), row=2, col=2)

    neon_layout(fig, title, height=920, width=1400)
    style_axes(fig, 1, 1, "w1", "w2")
    style_axes(fig, 1, 2, "PC1", "PC2")
    style_axes(fig, 2, 1, "Iteration", "Normalized magnitude")
    style_axes(fig, 2, 2, "Iteration", "Boundary score")
    return fig

fig3 = plot_representative_run(
    runs_3[rep_three],
    desc_3[rep_three],
    "Figure 3 — Representative Three-Phase Run: Geometry and Corrected Boundary Parser"
)
fig3.show()

# ------------------------------------------------------------
# 20. FIGURE 4 — EVENTS / TRANSITIONS / SUPERLEVELS
# ------------------------------------------------------------
def plot_regime_decomposition(desc, title):
    """
    Figure 4:
      - original structural energy
      - residual energy
      - corrected boundary score
      - structural profiles
      - events / transitions / superlevel boundaries
    """
    energy = np.asarray(desc["energy"], dtype=float)
    residual = np.asarray(desc["residual_energy"], dtype=float)
    bscore = np.asarray(desc["boundary_score"], dtype=float)

    event_intervals = desc["event_intervals"]
    transition_intervals = desc["transition_intervals"]
    super_boundaries = np.asarray(desc["superlevel_boundaries"], dtype=int)
    peaks = np.asarray(desc["boundary_peaks"], dtype=int)

    prof_p = np.asarray(desc["profile_param"], dtype=float)
    prof_f = np.asarray(desc["profile_func"], dtype=float)
    alpha = float(desc["alpha"])

    m = min(len(energy), len(residual), len(bscore), len(prof_p), len(prof_f))
    energy = energy[:m]
    residual = residual[:m]
    bscore = bscore[:m]
    prof_p = prof_p[:m]
    prof_f = prof_f[:m]
    super_boundaries = super_boundaries[super_boundaries < m]
    peaks = peaks[peaks < m]

    fig = make_subplots(
        rows=5, cols=1,
        shared_xaxes=True,
        subplot_titles=(
            "Original structural energy and superlevel threshold",
            "Residual energy",
            "Corrected multiscale boundary score",
            "Structural profile in parameter space",
            "Structural profile in function space"
        ),
        vertical_spacing=0.05
    )

    for a, b in transition_intervals:
        for row in [1, 2, 3, 4, 5]:
            fig.add_vrect(
                x0=a, x1=b + 0.5,
                fillcolor="rgba(110,247,255,0.10)",
                line_width=0,
                row=row, col=1
            )

    for a, b in event_intervals:
        for row in [1, 2, 3, 4, 5]:
            fig.add_vrect(
                x0=a, x1=b + 0.5,
                fillcolor="rgba(255,230,0,0.16)",
                line_width=0,
                row=row, col=1
            )

    for b in super_boundaries:
        for row in [1, 2, 3, 4, 5]:
            fig.add_vline(
                x=b,
                line=dict(color=NEON_PINK, width=1.4, dash="dot"),
                row=row, col=1
            )

    fig.add_trace(go.Scatter(
        x=np.arange(len(energy)),
        y=energy,
        mode="lines",
        line=dict(color=NEON_PURPLE, width=3),
        name="Structural energy"
    ), row=1, col=1)

    fig.add_hline(
        y=alpha,
        line=dict(color=NEON_YELLOW, width=2, dash="dash"),
        row=1, col=1
    )

    fig.add_trace(go.Scatter(
        x=np.arange(len(residual)),
        y=residual,
        mode="lines",
        line=dict(color=NEON_CYAN2, width=2),
        name="Residual energy"
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=np.arange(len(bscore)),
        y=bscore,
        mode="lines",
        line=dict(color=NEON_ORANGE, width=3),
        name="Boundary score"
    ), row=3, col=1)

    if len(peaks) > 0:
        fig.add_trace(go.Scatter(
            x=peaks,
            y=bscore[peaks],
            mode="markers",
            marker=dict(size=10, color=NEON_YELLOW, symbol="diamond"),
            name="Boundary peaks"
        ), row=3, col=1)

    fig.add_trace(go.Scatter(
        x=np.arange(len(prof_p)),
        y=prof_p,
        mode="lines",
        line=dict(color=NEON_GREEN, width=3),
        name="Profile param"
    ), row=4, col=1)

    fig.add_trace(go.Scatter(
        x=np.arange(len(prof_f)),
        y=prof_f,
        mode="lines",
        line=dict(color=NEON_BLUE, width=3),
        name="Profile func"
    ), row=5, col=1)

    neon_layout(fig, title, height=1350, width=1450)
    style_axes(fig, 1, 1, "Iteration", "Energy")
    style_axes(fig, 2, 1, "Iteration", "Residual")
    style_axes(fig, 3, 1, "Iteration", "Boundary score")
    style_axes(fig, 4, 1, "Iteration", "Profile param")
    style_axes(fig, 5, 1, "Iteration", "Profile func")
    return fig

fig4 = plot_regime_decomposition(
    desc_3[rep_three],
    "Figure 4 — Corrected Multiscale Boundary Parser: Events, Transitions, and Superlevel Geometry"
)
fig4.show()

# ------------------------------------------------------------
# 21. SENSITIVITY ANALYSIS
# ------------------------------------------------------------
def sensitivity_analysis(
    tag,
    schedule_fn,
    phase_boundaries,
    seeds,
    param_grid
):
    """
    Sensitivity analysis for the corrected boundary parser.
    Clean reporting version:
    the auxiliary diagnostics are intentionally omitted.
    """
    rows = []

    for cfg in param_grid:
        _, descs, df = run_family(
            tag=tag,
            schedule_fn=schedule_fn,
            seeds=seeds,
            phase_boundaries=phase_boundaries,
            **cfg
        )

        row = {
            "tag": tag,
            **cfg,
            "mean_n_events": float(df["n_events"].mean()),
            "std_n_events": float(df["n_events"].std()),
            "mean_n_boundaries": float(df["n_boundaries"].mean()),
            "mean_n_regimes": float(df["n_regimes"].mean()),
            "mean_n_transition_zones": float(df["n_transition_zones"].mean()),
            "mean_budget_ratio": float(df["budget_ratio"].mean()),
            "mean_boundary_loc_param": float(df["boundary_loc_param"].mean()),
            "mean_boundary_loc_func": float(df["boundary_loc_func"].mean()),
            "mean_boundary_peak_recall": float(df["boundary_peak_recall"].mean()),
            "mean_boundary_peak_precision": float(df["boundary_peak_precision"].mean()),
            "mean_mass_param": float(df["mass_param"].mean()),
            "mean_mass_func": float(df["mass_func"].mean())
        }
        rows.append(row)

    return pd.DataFrame(rows)

SENSITIVITY_GRID = [
    {
        **COMMON_EXTRACT_KWARGS,
        "q_boundary_peak": 0.88,
        "q_boundary_medium": 0.60,
        "min_peak_separation": 18,
        "max_events": 3
    },
    {
        **COMMON_EXTRACT_KWARGS,
        "q_boundary_peak": 0.91,
        "q_boundary_medium": 0.64,
        "min_peak_separation": 20,
        "max_events": 3
    },
    {
        **COMMON_EXTRACT_KWARGS,
        "q_boundary_peak": 0.93,
        "q_boundary_medium": 0.67,
        "min_peak_separation": 22,
        "max_events": 3
    }
]

sensitivity_three = sensitivity_analysis(
    tag="Three-phase dynamics",
    schedule_fn=three_phase_schedule,
    phase_boundaries=[45, 115],
    seeds=SEEDS,
    param_grid=SENSITIVITY_GRID
)

print("\n=== SENSITIVITY ANALYSIS — THREE-PHASE DYNAMICS ===")
print(sensitivity_three.round(6))

# ------------------------------------------------------------
# 22. FINAL INTERPRETIVE NOTES
# ------------------------------------------------------------
print("\n=== INTERPRETIVE NOTES FOR THE PAPER ===")
print(f"""
A) This computational layer is deliberately gauge-fixed.
   It does not claim exact optimization over the full quotient Q.

B) The key local objects are discrete windows acting as
   computational analogues of local jets.

C) The main admissible observables are:
   - oscillation,
   - triangular excess,
   - second-order window mass.

D) The canonical mesoscopic profiles are built as weighted sums of
   admissible local observables in parameter space and function space.

E) Structural masses operationalize the functional layer of the theory.

F) The original structural energy is preserved as the theory-aligned
   mesoscopic object from which superlevels and boundaries are built.

G) The corrected parser does NOT redefine Sections 2–6.
   It only replaces the macroscopic event logic:
   events are parsed from a multiscale structural boundary score,
   not from raw energy height alone.

H) The boundary score combines:
   - left/right contrast in structural profiles,
   - left/right contrast in residual energy,
   - fine-scale energy curvature.

I) Dominant events are transition regions around separated boundary peaks.
   Macro-regimes are then induced by those events.

J) The main evaluation layer used in the paper includes:
   - boundary-peak recall,
   - boundary-peak precision,
   - structural mass and concentration descriptors.

K) Table A = global geometry.
   Table B = structural profile + boundary evaluation summary.
   Table C = events / regimes / superlevel complexity.

L) Default settings:
   - jet_radius = {DEFAULT_JET_RADIUS}
   - smooth_window = {DEFAULT_SMOOTH_WINDOW}
   - alpha_quantile = {DEFAULT_ALPHA_QUANTILE}
   - lambda_min = {DEFAULT_LAMBDA_MIN}
   - osc_weight = {DEFAULT_OSC_WEIGHT}
   - exc_weight = {DEFAULT_EXC_WEIGHT}
   - som_weight = {DEFAULT_SOM_WEIGHT}
   - trend_window = {DEFAULT_TREND_WINDOW}
   - contrast_radius = {DEFAULT_CONTRAST_RADIUS}
   - q_boundary_peak = {DEFAULT_Q_BOUNDARY_PEAK}
   - q_boundary_medium = {DEFAULT_Q_BOUNDARY_MEDIUM}
   - min_peak_separation = {DEFAULT_MIN_PEAK_SEPARATION}
   - max_events = {DEFAULT_MAX_EVENTS}
   - burn_in_exclusion = {DEFAULT_BURN_IN_EXCLUSION}
   - max_half_width = {DEFAULT_MAX_HALF_WIDTH}
   - min_event_length = {DEFAULT_MIN_EVENT_LENGTH}

M) This script is intended to be the computational crown of the paper:
   not merely illustrative, but structurally faithful to Sections 2–6,
   with a mathematically sharper multiscale boundary parser and clean reporting.
""")

# ============================================================
# SECTION 7B — FINAL VALIDATION LAYER (CORRECTED FAST VERSION)
# ------------------------------------------------------------
# This reduced version preserves the main computational claims:
#   (D) multi-family discrimination
#   (E) temporal null control
#   (F) parser robustness
#
# Final methodological corrections
# --------------------------------
# 1) boundary_peak_recall and boundary_peak_precision are NOT
#    used as classification features in Tables D/E.
#    They depend on family-specific ground-truth boundaries.
# 2) These parser-vs-ground-truth quantities are retained ONLY
#    in Table F, where robustness of event recovery is the
#    actual object of study.
# 3) Optional plots are intentionally removed.
# 4) Execution remains faster than the original long version.
# ============================================================

# ------------------------------------------------------------
# 23. REGISTRY
# ------------------------------------------------------------
RUN_REGISTRY = {
    "Homogeneous dynamics": {
        "runs": runs_h,
        "descs": desc_h,
        "phase_boundaries": [],
        "expected_events": 0,
        "expected_regimes": 1,
    },
    "Regime-change dynamics": {
        "runs": runs_r,
        "descs": desc_r,
        "phase_boundaries": [T // 2],
        "expected_events": 1,
        "expected_regimes": 2,
    },
    "Three-phase dynamics": {
        "runs": runs_3,
        "descs": desc_3,
        "phase_boundaries": [45, 115],
        "expected_events": 2,
        "expected_regimes": 3,
    },
    "Pulse dynamics": {
        "runs": runs_p,
        "descs": desc_p,
        "phase_boundaries": [70, 90],
        "expected_events": 2,
        "expected_regimes": 3,
    },
    "Oscillatory dynamics": {
        "runs": runs_o,
        "descs": desc_o,
        "phase_boundaries": [],
        "expected_events": None,
        "expected_regimes": None,
    },
}

# ------------------------------------------------------------
# 24. FEATURE BLOCKS
# ------------------------------------------------------------
TERMINAL_FEATURES = [
    "final_loss",
    "final_acc",
]

GLOBAL_GEOMETRY_FEATURES = [
    "param_V",
    "func_V",
    "param_C",
    "func_C",
]

# CLEAN structural block:
# no direct parser-vs-ground-truth quantities here
STRUCTURAL_CORE_FEATURES_CLEAN = [
    "mass_param",
    "mass_func",
    "peak_param",
    "peak_func",
    "boundary_loc_param",
    "boundary_loc_func",
    "n_events",
    "n_boundaries",
    "n_regimes",
    "n_regime_changes",
    "n_transition_zones",
    "mean_transition_length",
    "n_superlevel_components",
    "n_superlevel_boundaries",
    "budget_ratio",
    "max_energy",
    "max_residual_energy",
    "max_profile_param",
    "max_profile_func",
]

FEATURE_BLOCKS = {
    "Terminal only": TERMINAL_FEATURES,
    "Global geometry": GLOBAL_GEOMETRY_FEATURES,
    "Structural core (clean)": STRUCTURAL_CORE_FEATURES_CLEAN,
    "Global + structural (clean)": GLOBAL_GEOMETRY_FEATURES + STRUCTURAL_CORE_FEATURES_CLEAN,
    "Terminal + global + structural (clean)": TERMINAL_FEATURES + GLOBAL_GEOMETRY_FEATURES + STRUCTURAL_CORE_FEATURES_CLEAN,
}

# ------------------------------------------------------------
# 25. FAST VALIDATION SETTINGS
# ------------------------------------------------------------
FAST_CV_SPLITS_MAIN = 5
FAST_CV_REPEATS_MAIN = 10

FAST_CV_SPLITS_FAMILY = 5
FAST_CV_REPEATS_FAMILY = 5

N_SURROGATES_PER_RUN = 2

# ------------------------------------------------------------
# 26. VALIDATION HELPERS
# ------------------------------------------------------------
def scalarize_desc_list(desc_list):
    """
    Keep only scalar / text fields from descriptor dictionaries.
    Arrays, lists and nested objects are dropped.
    """
    rows = []
    for d in desc_list:
        row = {}
        for k, v in d.items():
            if isinstance(v, (np.ndarray, list, dict)):
                continue
            row[k] = v
        rows.append(row)
    return pd.DataFrame(rows)

def available_features(df, feature_cols):
    """
    Keep only features that are actually present in the DataFrame.
    """
    return [c for c in feature_cols if c in df.columns]

def prepare_matrix(df, feature_cols):
    """
    Convert a dataframe block into a clean numeric matrix.
    """
    cols = available_features(df, feature_cols)
    X = df[cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X.astype(float).values, cols

def evaluate_classifier(df, feature_cols, target_col, n_splits=5, n_repeats=10, random_state=123):
    """
    Repeated stratified CV with a simple logistic model.
    The classifier remains intentionally simple:
    if performance is strong here, the structure is genuinely informative.
    """
    X, used_cols = prepare_matrix(df, feature_cols)
    y = df[target_col].astype(str).values

    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=random_state
    )

    fold_rows = []

    for train_idx, test_idx in cv.split(X, y):
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=4000,
                class_weight="balanced",
                solver="lbfgs"
            ))
        ])

        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[test_idx])

        fold_rows.append({
            "accuracy": accuracy_score(y[test_idx], pred),
            "balanced_accuracy": balanced_accuracy_score(y[test_idx], pred),
            "macro_f1": f1_score(y[test_idx], pred, average="macro"),
        })

    fold_df = pd.DataFrame(fold_rows)
    out = {
        "accuracy_mean": fold_df["accuracy"].mean(),
        "accuracy_std": fold_df["accuracy"].std(),
        "balanced_accuracy_mean": fold_df["balanced_accuracy"].mean(),
        "balanced_accuracy_std": fold_df["balanced_accuracy"].std(),
        "macro_f1_mean": fold_df["macro_f1"].mean(),
        "macro_f1_std": fold_df["macro_f1"].std(),
        "n_features": len(used_cols),
        "used_features": ", ".join(used_cols),
    }
    return out

def time_scramble_index(n, rng):
    """
    Destroy internal temporal order while preserving endpoints.
    """
    if n <= 2:
        return np.arange(n)

    middle = np.arange(1, n - 1)
    middle = rng.permutation(middle)
    return np.r_[0, middle, n - 1]

def make_temporal_surrogate(run, rng):
    """
    Build a temporal surrogate of a run:
    - param_path and func_path are scrambled with the same internal permutation
    - endpoints are preserved
    - terminal losses / accuracies remain unchanged
    """
    idx = time_scramble_index(len(run["param_path"]), rng)

    surrogate = {}
    for k, v in run.items():
        if isinstance(v, np.ndarray):
            surrogate[k] = v.copy()
        else:
            surrogate[k] = deepcopy(v)

    surrogate["param_path"] = run["param_path"][idx].copy()
    surrogate["func_path"] = run["func_path"][idx].copy()
    surrogate["losses"] = run["losses"].copy()
    surrogate["accs"] = run["accs"].copy()

    return surrogate

# ------------------------------------------------------------
# 27. TABLE D — MULTI-FAMILY DISCRIMINATION
# ------------------------------------------------------------
multi_family_rows = []

for block_name, feature_cols in FEATURE_BLOCKS.items():
    res = evaluate_classifier(
        df=summary_df,
        feature_cols=feature_cols,
        target_col="tag",
        n_splits=FAST_CV_SPLITS_MAIN,
        n_repeats=FAST_CV_REPEATS_MAIN,
        random_state=123
    )
    res["feature_block"] = block_name
    multi_family_rows.append(res)

table_D = (
    pd.DataFrame(multi_family_rows)
    .set_index("feature_block")
    .sort_values("balanced_accuracy_mean", ascending=False)
    .round(4)
)

print("\n=== TABLE D: MULTI-FAMILY DISCRIMINATION ===")
print(table_D[[
    "accuracy_mean", "accuracy_std",
    "balanced_accuracy_mean", "balanced_accuracy_std",
    "macro_f1_mean", "macro_f1_std",
    "n_features"
]])

print("\n=== TABLE D: FEATURES USED ===")
for idx, row in table_D.iterrows():
    print(f"{idx}: {row['used_features']}")

# ------------------------------------------------------------
# 28. TABLE E — TEMPORAL NULL CONTROL
# ------------------------------------------------------------
rng_null = np.random.default_rng(2026)
temporal_rows = []

for tag, info in RUN_REGISTRY.items():
    original_scalar_df = scalarize_desc_list(info["descs"])

    # Replicate originals to keep original/scrambled classes balanced.
    for _, row in original_scalar_df.iterrows():
        for _ in range(N_SURROGATES_PER_RUN):
            row_dict = row.to_dict()
            row_dict["temporal_label"] = "original"
            row_dict["family_tag"] = tag
            temporal_rows.append(row_dict)

    # Temporal surrogates.
    for run in info["runs"]:
        for _ in range(N_SURROGATES_PER_RUN):
            surrogate_run = make_temporal_surrogate(run, rng_null)
            surrogate_desc = extract_structural_descriptors(
                surrogate_run,
                phase_boundaries=info["phase_boundaries"],
                **COMMON_EXTRACT_KWARGS
            )
            surrogate_row = {
                k: v for k, v in surrogate_desc.items()
                if not isinstance(v, (np.ndarray, list, dict))
            }
            surrogate_row["temporal_label"] = "scrambled"
            surrogate_row["family_tag"] = tag
            temporal_rows.append(surrogate_row)

temporal_control_df = pd.DataFrame(temporal_rows)

temporal_eval_rows = []
for block_name, feature_cols in FEATURE_BLOCKS.items():
    res = evaluate_classifier(
        df=temporal_control_df,
        feature_cols=feature_cols,
        target_col="temporal_label",
        n_splits=FAST_CV_SPLITS_MAIN,
        n_repeats=FAST_CV_REPEATS_MAIN,
        random_state=456
    )
    res["feature_block"] = block_name
    temporal_eval_rows.append(res)

table_E = (
    pd.DataFrame(temporal_eval_rows)
    .set_index("feature_block")
    .sort_values("balanced_accuracy_mean", ascending=False)
    .round(4)
)

print("\n=== TABLE E: TEMPORAL NULL CONTROL (ORIGINAL vs SCRAMBLED) ===")
print(table_E[[
    "accuracy_mean", "accuracy_std",
    "balanced_accuracy_mean", "balanced_accuracy_std",
    "macro_f1_mean", "macro_f1_std",
    "n_features"
]])

print("\n=== TABLE E: FEATURES USED ===")
for idx, row in table_E.iterrows():
    print(f"{idx}: {row['used_features']}")

# Family-wise temporal null control
family_temporal_rows = []
for family_tag in temporal_control_df["family_tag"].unique():
    df_family = temporal_control_df[temporal_control_df["family_tag"] == family_tag].copy()

    for block_name, feature_cols in FEATURE_BLOCKS.items():
        res = evaluate_classifier(
            df=df_family,
            feature_cols=feature_cols,
            target_col="temporal_label",
            n_splits=FAST_CV_SPLITS_FAMILY,
            n_repeats=FAST_CV_REPEATS_FAMILY,
            random_state=789
        )
        res["family_tag"] = family_tag
        res["feature_block"] = block_name
        family_temporal_rows.append(res)

table_E_family = (
    pd.DataFrame(family_temporal_rows)
    .set_index(["family_tag", "feature_block"])
    .sort_values(["family_tag", "balanced_accuracy_mean"], ascending=[True, False])
    .round(4)
)

print("\n=== TABLE E-FAMILY: WITHIN-FAMILY TEMPORAL NULL CONTROL ===")
print(table_E_family[[
    "accuracy_mean", "accuracy_std",
    "balanced_accuracy_mean", "balanced_accuracy_std",
    "macro_f1_mean", "macro_f1_std",
    "n_features"
]])

# ------------------------------------------------------------
# 29. TABLE F — PARSER ROBUSTNESS (REDUCED GRID)
# ------------------------------------------------------------
PARSER_STRESS_CONFIGS = [
    {
        "config_name": "default",
        "q_boundary_peak": DEFAULT_Q_BOUNDARY_PEAK,
        "q_boundary_medium": DEFAULT_Q_BOUNDARY_MEDIUM,
        "min_peak_separation": DEFAULT_MIN_PEAK_SEPARATION,
        "max_half_width": DEFAULT_MAX_HALF_WIDTH,
        "min_event_length": DEFAULT_MIN_EVENT_LENGTH,
        "trend_window": DEFAULT_TREND_WINDOW,
        "contrast_radius": DEFAULT_CONTRAST_RADIUS,
    },
    {
        "config_name": "looser_thresholds",
        "q_boundary_peak": 0.88,
        "q_boundary_medium": 0.60,
        "min_peak_separation": DEFAULT_MIN_PEAK_SEPARATION,
        "max_half_width": DEFAULT_MAX_HALF_WIDTH,
        "min_event_length": DEFAULT_MIN_EVENT_LENGTH,
        "trend_window": DEFAULT_TREND_WINDOW,
        "contrast_radius": DEFAULT_CONTRAST_RADIUS,
    },
    {
        "config_name": "stricter_thresholds",
        "q_boundary_peak": 0.94,
        "q_boundary_medium": 0.68,
        "min_peak_separation": DEFAULT_MIN_PEAK_SEPARATION,
        "max_half_width": DEFAULT_MAX_HALF_WIDTH,
        "min_event_length": DEFAULT_MIN_EVENT_LENGTH,
        "trend_window": DEFAULT_TREND_WINDOW,
        "contrast_radius": DEFAULT_CONTRAST_RADIUS,
    },
    {
        "config_name": "shorter_trend_context",
        "q_boundary_peak": DEFAULT_Q_BOUNDARY_PEAK,
        "q_boundary_medium": DEFAULT_Q_BOUNDARY_MEDIUM,
        "min_peak_separation": DEFAULT_MIN_PEAK_SEPARATION,
        "max_half_width": DEFAULT_MAX_HALF_WIDTH,
        "min_event_length": DEFAULT_MIN_EVENT_LENGTH,
        "trend_window": 25,
        "contrast_radius": 6,
    },
]

robustness_rows = []

for cfg in PARSER_STRESS_CONFIGS:
    for tag, info in RUN_REGISTRY.items():
        for run in info["runs"]:
            kwargs = dict(COMMON_EXTRACT_KWARGS)
            kwargs.update({
                "q_boundary_peak": cfg["q_boundary_peak"],
                "q_boundary_medium": cfg["q_boundary_medium"],
                "min_peak_separation": cfg["min_peak_separation"],
                "max_half_width": cfg["max_half_width"],
                "min_event_length": cfg["min_event_length"],
                "trend_window": cfg["trend_window"],
                "contrast_radius": cfg["contrast_radius"],
            })

            d = extract_structural_descriptors(
                run,
                phase_boundaries=info["phase_boundaries"],
                **kwargs
            )

            exact_event_recovery = np.nan
            exact_regime_recovery = np.nan
            exact_structural_recovery = np.nan

            if info["expected_events"] is not None:
                exact_event_recovery = float(d["n_events"] == info["expected_events"])

            if info["expected_regimes"] is not None:
                exact_regime_recovery = float(d["n_regimes"] == info["expected_regimes"])

            if (info["expected_events"] is not None) and (info["expected_regimes"] is not None):
                exact_structural_recovery = float(
                    (d["n_events"] == info["expected_events"]) and
                    (d["n_regimes"] == info["expected_regimes"])
                )

            robustness_rows.append({
                "config_name": cfg["config_name"],
                "tag": tag,
                "n_true_boundaries": len(info["phase_boundaries"]),
                "boundary_peak_recall": d["boundary_peak_recall"],
                "boundary_peak_precision": d["boundary_peak_precision"],
                "n_events": d["n_events"],
                "n_regimes": d["n_regimes"],
                "exact_event_recovery": exact_event_recovery,
                "exact_regime_recovery": exact_regime_recovery,
                "exact_structural_recovery": exact_structural_recovery,
            })

robustness_df = pd.DataFrame(robustness_rows)

exact_summary = (
    robustness_df[robustness_df["exact_structural_recovery"].notna()]
    .groupby("config_name")[[
        "exact_event_recovery",
        "exact_regime_recovery",
        "exact_structural_recovery"
    ]]
    .mean()
)

boundary_summary = (
    robustness_df[robustness_df["n_true_boundaries"] > 0]
    .groupby("config_name")[[
        "boundary_peak_recall",
        "boundary_peak_precision"
    ]]
    .mean()
)

table_F = exact_summary.join(boundary_summary, how="left")
table_F["robustness_score"] = (
    0.50 * table_F["exact_structural_recovery"] +
    0.25 * table_F["boundary_peak_recall"] +
    0.25 * table_F["boundary_peak_precision"]
)

table_F = table_F.sort_values("robustness_score", ascending=False).round(4)

print("\n=== TABLE F: PARSER ROBUSTNESS SUMMARY ===")
print(table_F)

table_F_family = (
    robustness_df
    .groupby(["config_name", "tag"])[[
        "exact_event_recovery",
        "exact_regime_recovery",
        "exact_structural_recovery",
        "boundary_peak_recall",
        "boundary_peak_precision"
    ]]
    .mean()
    .round(4)
)

print("\n=== TABLE F-FAMILY: PARSER ROBUSTNESS BY FAMILY ===")
print(table_F_family)

# ------------------------------------------------------------
# 30. INTERPRETIVE GUIDE
# ------------------------------------------------------------
print("\n=== INTERPRETIVE GUIDE ===")
print("1) Table D:")
print("   If the clean structural blocks clearly beat 'Terminal only',")
print("   then the main paper claim is computationally supported without")
print("   using parser-to-ground-truth leakage features.")
print("")
print("2) Table E:")
print("   If 'Terminal only' stays near chance while the clean structural")
print("   blocks remain clearly above chance, then the method captures")
print("   temporal organization rather than endpoints only.")
print("")
print("3) Table F:")
print("   If the default parser remains near the top, and nearby stress")
print("   configurations perform similarly, the parser is not an unstable")
print("   ad hoc device.")
print("")
print("4) Main narrative preserved in this corrected fast block:")
print("   - terminal metrics alone are insufficient;")
print("   - clean structural descriptors recover family-level organization;")
print("   - scrambling internal temporal order damages detectability;")
print("   - core conclusions persist under moderate parser perturbations.")


=== AUDIT OF BOUNDED STRUCTURAL INDICES ===
peak_param: min=0.262323, max=0.688564
peak_func: min=0.325986, max=0.833159
boundary_loc_param: min=0.000000, max=0.186351
boundary_loc_func: min=0.000000, max=0.137279
boundary_peak_recall: min=0.000000, max=1.000000
boundary_peak_precision: min=0.000000, max=1.000000
All bounded structural indices lie in [0, 1].

=== PAPER TABLE A: GLOBAL GEOMETRIC SUMMARY ===
                         param_V              func_V             param_C  \
                            mean       std      mean       std      mean   
tag                                                                        
Homogeneous dynamics    2.341795  0.089689  8.777722  0.408981  0.078132   
Oscillatory dynamics    2.497847  0.130742  8.989847  0.449464  0.117427   
Pulse dynamics          2.392670  0.137967  8.858310  0.474016  0.115130   
Regime-change dynamics  2.524541  0.123002  9.040072  0.422428  0.105454   
Three-phase dynamics    2.411153  0.101897  8.877821  0.4


=== SENSITIVITY ANALYSIS — THREE-PHASE DYNAMICS ===
                    tag  jet_radius  smooth_window  alpha_quantile  \
0  Three-phase dynamics           6             11            0.85   
1  Three-phase dynamics           6             11            0.85   
2  Three-phase dynamics           6             11            0.85   

   lambda_min  osc_weight  exc_weight  som_weight  trend_window  \
0           8         0.2         1.0         1.0            31   
1           8         0.2         1.0         1.0            31   
2           8         0.2         1.0         1.0            31   

   contrast_radius  ...  mean_n_boundaries  mean_n_regimes  \
0                8  ...                1.1            2.05   
1                8  ...                3.0            3.00   
2                8  ...                3.0            3.00   

   mean_n_transition_zones  mean_budget_ratio  mean_boundary_loc_param  \
0                     1.05           13.12855                 0.079253   


In [6]:
# ============================================================
# SECTION 7C-FINAL — COMPACT EXTERNAL + MINI-PDE VALIDATION
# ------------------------------------------------------------
# Main-text tables kept in this compact version:
#   Table G  : External benchmark summary (Airfoil only)
#   Table I  : External temporal null control (Airfoil only)
#   Table J  : Mini-PDEBench Darcy summary
#   Table L  : Mini-PDEBench Darcy temporal null control
#   Table M  : Compact cross-case evidence summary
#
# Editorial purpose:
#   Keep only the strongest evidence that clearly elevates the paper:
#   - real external validation
#   - PDE-based scientific ML validation
#   - temporal scrambling weakens terminal-only summaries
#   - geometric/structural descriptors remain informative
#
# Assumption:
#   The notebook has already defined:
#     - extract_structural_descriptors(...)
#     - COMMON_EXTRACT_KWARGS
# ============================================================

# ------------------------------------------------------------
# 31. IMPORTS
# ------------------------------------------------------------
import os
import urllib.request
from copy import deepcopy

import h5py
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, accuracy_score, balanced_accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ------------------------------------------------------------
# 32. GLOBAL CONFIG
# ------------------------------------------------------------
DATA_DIR = "./external_public_datasets"
os.makedirs(DATA_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ----------------------------
# Airfoil (kept as main real external case)
# ----------------------------
AIRFOIL_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00291/airfoil_self_noise.dat"
AIRFOIL_PATH = os.path.join(DATA_DIR, "airfoil_self_noise.dat")

AIRFOIL_SEEDS = (123, 124, 125, 126)
AIRFOIL_EPOCHS = 40
AIRFOIL_BATCH_SIZE = 64
AIRFOIL_HIDDEN_DIM = 64
AIRFOIL_SGD_LR = 0.03
AIRFOIL_PROBE_SIZE = 128

# ----------------------------
# Mini-PDEBench Darcy
# ----------------------------
DARCY_URL = "https://darus.uni-stuttgart.de/api/access/datafile/133219"
DARCY_PATH = os.path.join(DATA_DIR, "2D_DarcyFlow_beta1.0_Train.hdf5")

# RAM-safe defaults
DARCY_MAX_SAMPLES = 96
DARCY_DOWNSAMPLE_STEP = 4
DARCY_PROBE_SIZE = 8
DARCY_EPOCHS = 25
DARCY_BATCH_SIZE = 16
DARCY_HIDDEN_DIM = 128
DARCY_SGD_LR = 0.02
DARCY_SEEDS = (123, 124, 125, 126)

# ------------------------------------------------------------
# 33. DOWNLOAD HELPERS
# ------------------------------------------------------------
def download_file(url, filepath):
    if os.path.exists(filepath):
        print(f"Using cached file: {filepath}")
        return
    print(f"Downloading: {url}")
    urllib.request.urlretrieve(url, filepath)

def download_large_file(url, filepath, chunk_size=1024 * 1024):
    """
    Chunked download for large public files.
    """
    if os.path.exists(filepath):
        print(f"Using cached file: {filepath}")
        return

    print(f"Downloading large file:\n  {url}\n  -> {filepath}")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

    with urllib.request.urlopen(req) as response, open(filepath, "wb") as f:
        total = response.headers.get("Content-Length")
        total = int(total) if total is not None else None
        downloaded = 0

        while True:
            chunk = response.read(chunk_size)
            if not chunk:
                break
            f.write(chunk)
            downloaded += len(chunk)

            if total is not None:
                print(
                    f"\rDownloaded {downloaded / 1024**3:.2f} / {total / 1024**3:.2f} GB",
                    end=""
                )
            else:
                print(f"\rDownloaded {downloaded / 1024**3:.2f} GB", end="")

    print("\nDownload complete.")

# ------------------------------------------------------------
# 34. DATA DOWNLOAD
# ------------------------------------------------------------
download_file(AIRFOIL_URL, AIRFOIL_PATH)
download_large_file(DARCY_URL, DARCY_PATH)

# ------------------------------------------------------------
# 35. DATA LOADERS
# ------------------------------------------------------------
def load_airfoil_self_noise_dataset(data_path):
    colnames = [
        "frequency",
        "angle_of_attack",
        "chord_length",
        "free_stream_velocity",
        "suction_side_displacement_thickness",
        "scaled_sound_pressure_level",
    ]
    df = pd.read_csv(data_path, sep=r"\s+", header=None, names=colnames)

    y = df["scaled_sound_pressure_level"].astype(np.float32).to_numpy().reshape(-1, 1)
    X = df.drop(columns=["scaled_sound_pressure_level"]).astype(np.float32).to_numpy()
    return X, y

def _to_2d_field(arr):
    z = np.asarray(arr, dtype=np.float32)
    z = np.squeeze(z)
    if z.ndim != 2:
        raise ValueError(f"Expected a 2D field after squeeze, got shape={z.shape}")
    return z

def spatial_downsample(field_2d, step=4):
    return field_2d[::step, ::step].copy()

def load_darcy_dataset_ram_safe(h5_path, max_samples=96, downsample_step=4):
    """
    Loads a compact Darcy subset:
      X = flattened permeability field (nu)
      y = flattened solution field (tensor)
    """
    X_list, y_list = [], []

    with h5py.File(h5_path, "r") as h5f:
        print("Available HDF5 keys:", list(h5f.keys()))
        tensor_ds = h5f["tensor"]
        nu_ds = h5f["nu"]

        n_total = len(tensor_ds)
        n_use = min(max_samples, n_total)

        for i in range(n_use):
            x_field = _to_2d_field(nu_ds[i])
            y_field = _to_2d_field(tensor_ds[i])

            x_small = spatial_downsample(x_field, step=downsample_step)
            y_small = spatial_downsample(y_field, step=downsample_step)

            X_list.append(x_small.reshape(-1))
            y_list.append(y_small.reshape(-1))

    X = np.stack(X_list).astype(np.float32)
    y = np.stack(y_list).astype(np.float32)

    print(f"Loaded Darcy subset: X={X.shape}, y={y.shape}")
    print(f"Spatial downsample step: {downsample_step}")
    return X, y

X_airfoil, y_airfoil = load_airfoil_self_noise_dataset(AIRFOIL_PATH)
X_darcy, y_darcy = load_darcy_dataset_ram_safe(
    DARCY_PATH,
    max_samples=DARCY_MAX_SAMPLES,
    downsample_step=DARCY_DOWNSAMPLE_STEP
)

print("\nLoaded datasets:")
print("Airfoil:", X_airfoil.shape, y_airfoil.shape)
print("Mini-Darcy:", X_darcy.shape, y_darcy.shape)

# ------------------------------------------------------------
# 36. SPLITS
# ------------------------------------------------------------
def make_regression_splits(X, y, random_state=123, probe_size=128):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    if y.ndim == 1:
        y = y.reshape(-1, 1)

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.20, random_state=random_state
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, random_state=random_state
    )
    # 60 / 20 / 20

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train_s = x_scaler.fit_transform(X_train)
    X_val_s   = x_scaler.transform(X_val)
    X_test_s  = x_scaler.transform(X_test)

    y_train_s = y_scaler.fit_transform(y_train)
    y_val_s   = y_scaler.transform(y_val)
    y_test_s  = y_scaler.transform(y_test)

    probe_n = min(probe_size, len(X_val_s))
    X_probe_s = X_val_s[:probe_n].copy()

    return {
        "X_train": X_train_s.astype(np.float32),
        "y_train": y_train_s.astype(np.float32),
        "X_val": X_val_s.astype(np.float32),
        "y_val": y_val_s.astype(np.float32),
        "X_test": X_test_s.astype(np.float32),
        "y_test": y_test_s.astype(np.float32),
        "X_probe": X_probe_s.astype(np.float32),
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
    }

splits_airfoil = make_regression_splits(
    X_airfoil, y_airfoil, random_state=123, probe_size=AIRFOIL_PROBE_SIZE
)
splits_darcy = make_regression_splits(
    X_darcy, y_darcy, random_state=123, probe_size=DARCY_PROBE_SIZE
)

# ------------------------------------------------------------
# 37. GENERIC REGRESSION MODEL
# ------------------------------------------------------------
class FlexibleMLPRegressor(nn.Module):
    def __init__(self, in_dim, out_dim=1, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

def flatten_parameters(model):
    return torch.cat([p.detach().view(-1).cpu() for p in model.parameters()]).numpy()

@torch.no_grad()
def function_signature(model, X_probe_tensor):
    pred = model(X_probe_tensor).detach().cpu().numpy()
    return pred.reshape(-1)

def create_optimizer_and_scheduler(model, condition_name, epochs, sgd_lr=0.03):
    name = condition_name.lower()

    if name == "sgd-constant":
        optimizer = torch.optim.SGD(model.parameters(), lr=sgd_lr, momentum=0.9)
        scheduler = None

    elif name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        scheduler = None

    elif name == "cosine-annealing":
        optimizer = torch.optim.SGD(model.parameters(), lr=sgd_lr, momentum=0.9)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    else:
        raise ValueError(f"Unknown condition: {condition_name}")

    return optimizer, scheduler

def train_one_run(
    dataset_splits,
    condition_name,
    seed=123,
    epochs=40,
    batch_size=64,
    hidden_dim=64,
    sgd_lr=0.03
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    X_train = torch.tensor(dataset_splits["X_train"], dtype=torch.float32, device=DEVICE)
    y_train = torch.tensor(dataset_splits["y_train"], dtype=torch.float32, device=DEVICE)
    X_val   = torch.tensor(dataset_splits["X_val"], dtype=torch.float32, device=DEVICE)
    y_val   = torch.tensor(dataset_splits["y_val"], dtype=torch.float32, device=DEVICE)
    X_test  = torch.tensor(dataset_splits["X_test"], dtype=torch.float32, device=DEVICE)
    y_test  = torch.tensor(dataset_splits["y_test"], dtype=torch.float32, device=DEVICE)
    X_probe = torch.tensor(dataset_splits["X_probe"], dtype=torch.float32, device=DEVICE)

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=min(batch_size, len(X_train)),
        shuffle=True
    )

    model = FlexibleMLPRegressor(
        in_dim=X_train.shape[1],
        out_dim=y_train.shape[1],
        hidden_dim=hidden_dim
    ).to(DEVICE)

    criterion = nn.MSELoss()
    optimizer, scheduler = create_optimizer_and_scheduler(
        model, condition_name, epochs, sgd_lr=sgd_lr
    )

    losses = []
    param_path = []
    func_path = []

    for epoch in range(epochs):
        model.train()

        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val).item()

        losses.append(val_loss)
        param_path.append(flatten_parameters(model))
        func_path.append(function_signature(model, X_probe))

    model.eval()
    with torch.no_grad():
        test_pred = model(X_test).detach().cpu().numpy()

    y_test_np = y_test.detach().cpu().numpy()
    final_score = r2_score(y_test_np.reshape(-1), test_pred.reshape(-1))
    losses_arr = np.asarray(losses, dtype=np.float32)

    run = {
        "param_path": np.vstack(param_path).astype(np.float32),
        "func_path": np.vstack(func_path).astype(np.float32),
        "losses": losses_arr,
        "accs": (-losses_arr),  # compatibility proxy
        "final_score": float(final_score),
        "condition_name": condition_name,
        "seed": int(seed),
        "epochs": int(epochs),
    }
    return run

def train_condition_runs(
    dataset_splits,
    condition_name,
    seeds=(123, 124),
    epochs=40,
    batch_size=64,
    hidden_dim=64,
    sgd_lr=0.03
):
    runs = []
    for seed in seeds:
        print(f"Training condition={condition_name}, seed={seed}")
        run = train_one_run(
            dataset_splits=dataset_splits,
            condition_name=condition_name,
            seed=seed,
            epochs=epochs,
            batch_size=batch_size,
            hidden_dim=hidden_dim,
            sgd_lr=sgd_lr
        )
        runs.append(run)
    return runs

# ------------------------------------------------------------
# 38. RUN AIRFOIL + MINI-DARCY
# ------------------------------------------------------------
runs_airfoil_sgd = train_condition_runs(
    splits_airfoil,
    "SGD-constant",
    seeds=AIRFOIL_SEEDS,
    epochs=AIRFOIL_EPOCHS,
    batch_size=AIRFOIL_BATCH_SIZE,
    hidden_dim=AIRFOIL_HIDDEN_DIM,
    sgd_lr=AIRFOIL_SGD_LR
)

runs_airfoil_adam = train_condition_runs(
    splits_airfoil,
    "Adam",
    seeds=AIRFOIL_SEEDS,
    epochs=AIRFOIL_EPOCHS,
    batch_size=AIRFOIL_BATCH_SIZE,
    hidden_dim=AIRFOIL_HIDDEN_DIM,
    sgd_lr=AIRFOIL_SGD_LR
)

runs_darcy_sgd = train_condition_runs(
    splits_darcy,
    "SGD-constant",
    seeds=DARCY_SEEDS,
    epochs=DARCY_EPOCHS,
    batch_size=DARCY_BATCH_SIZE,
    hidden_dim=DARCY_HIDDEN_DIM,
    sgd_lr=DARCY_SGD_LR
)

runs_darcy_adam = train_condition_runs(
    splits_darcy,
    "Adam",
    seeds=DARCY_SEEDS,
    epochs=DARCY_EPOCHS,
    batch_size=DARCY_BATCH_SIZE,
    hidden_dim=DARCY_HIDDEN_DIM,
    sgd_lr=DARCY_SGD_LR
)

COMPACT_BENCHMARKS = {
    "Airfoil-Self-Noise": {
        "phase_boundaries": [],
        "conditions": {
            "SGD-constant": runs_airfoil_sgd,
            "Adam": runs_airfoil_adam,
        },
    },
    "Mini-PDEBench-Darcy": {
        "phase_boundaries": [],
        "conditions": {
            "SGD-constant": runs_darcy_sgd,
            "Adam": runs_darcy_adam,
        },
    },
}

# ------------------------------------------------------------
# 39. FEATURE HELPERS
# ------------------------------------------------------------
if "available_features" not in globals():
    def available_features(df, feature_cols):
        return [c for c in feature_cols if c in df.columns]

if "prepare_matrix" not in globals():
    def prepare_matrix(df, feature_cols):
        cols = available_features(df, feature_cols)
        X = df[cols].copy()
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return X.astype(float).values, cols

if "evaluate_classifier" not in globals():
    def evaluate_classifier(df, feature_cols, target_col, n_splits=2, n_repeats=10, random_state=123):
        X, used_cols = prepare_matrix(df, feature_cols)
        y = df[target_col].astype(str).values

        cv = RepeatedStratifiedKFold(
            n_splits=n_splits,
            n_repeats=n_repeats,
            random_state=random_state
        )

        fold_rows = []

        for train_idx, test_idx in cv.split(X, y):
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                    max_iter=4000,
                    class_weight="balanced",
                    solver="lbfgs"
                ))
            ])
            model.fit(X[train_idx], y[train_idx])
            pred = model.predict(X[test_idx])

            fold_rows.append({
                "accuracy": accuracy_score(y[test_idx], pred),
                "balanced_accuracy": balanced_accuracy_score(y[test_idx], pred),
                "macro_f1": f1_score(y[test_idx], pred, average="macro"),
            })

        fold_df = pd.DataFrame(fold_rows)
        return {
            "accuracy_mean": fold_df["accuracy"].mean(),
            "accuracy_std": fold_df["accuracy"].std(),
            "balanced_accuracy_mean": fold_df["balanced_accuracy"].mean(),
            "balanced_accuracy_std": fold_df["balanced_accuracy"].std(),
            "macro_f1_mean": fold_df["macro_f1"].mean(),
            "macro_f1_std": fold_df["macro_f1"].std(),
            "n_features": len(used_cols),
            "used_features": ", ".join(used_cols),
        }

if "time_scramble_index" not in globals():
    def time_scramble_index(n, rng):
        if n <= 2:
            return np.arange(n)
        middle = np.arange(1, n - 1)
        middle = rng.permutation(middle)
        return np.r_[0, middle, n - 1]

def make_temporal_surrogate(run, rng):
    idx = time_scramble_index(len(run["param_path"]), rng)

    surrogate = {}
    for k, v in run.items():
        if isinstance(v, np.ndarray):
            surrogate[k] = v.copy()
        else:
            surrogate[k] = deepcopy(v)

    surrogate["param_path"] = run["param_path"][idx].copy()
    surrogate["func_path"] = run["func_path"][idx].copy()
    surrogate["losses"] = run["losses"].copy()
    surrogate["accs"] = run["accs"].copy()
    surrogate["final_score"] = run["final_score"]

    return surrogate

def make_run_compatible(run, tag_name):
    compat_run = deepcopy(run)
    compat_run["tag"] = tag_name
    if "accs" not in compat_run:
        losses = np.asarray(compat_run["losses"], dtype=np.float32)
        compat_run["accs"] = (-losses).astype(np.float32)
    return compat_run

def extract_desc(run, phase_boundaries=None, tag_name="external"):
    if phase_boundaries is None:
        phase_boundaries = []

    compat_run = make_run_compatible(run, tag_name=tag_name)

    d = extract_structural_descriptors(
        compat_run,
        phase_boundaries=phase_boundaries,
        **COMMON_EXTRACT_KWARGS
    )

    row = {}
    for k, v in d.items():
        if isinstance(v, (np.ndarray, list, dict)):
            continue
        row[k] = v

    row["final_loss"] = float(run["losses"][-1])
    row["final_score"] = float(run["final_score"])

    if "final_acc" in row:
        del row["final_acc"]

    return row

# ------------------------------------------------------------
# 40. CLEAN FEATURE BLOCKS
# ------------------------------------------------------------
TERMINAL_FEATURES = [
    "final_loss",
    "final_score",
]

GLOBAL_GEOMETRY_FEATURES = [
    "param_V",
    "func_V",
    "param_C",
    "func_C",
]

STRUCTURAL_CORE_FEATURES_CLEAN = [
    "mass_param",
    "mass_func",
    "peak_param",
    "peak_func",
    "boundary_loc_param",
    "boundary_loc_func",
    "n_events",
    "n_boundaries",
    "n_regimes",
    "n_regime_changes",
    "n_transition_zones",
    "mean_transition_length",
    "n_superlevel_components",
    "n_superlevel_boundaries",
    "budget_ratio",
    "max_energy",
    "max_residual_energy",
    "max_profile_param",
    "max_profile_func",
]

FEATURE_BLOCKS = {
    "Terminal only": TERMINAL_FEATURES,
    "Global geometry": GLOBAL_GEOMETRY_FEATURES,
    "Structural core (clean)": STRUCTURAL_CORE_FEATURES_CLEAN,
    "Global + structural (clean)": GLOBAL_GEOMETRY_FEATURES + STRUCTURAL_CORE_FEATURES_CLEAN,
    "Terminal + global + structural (clean)": TERMINAL_FEATURES + GLOBAL_GEOMETRY_FEATURES + STRUCTURAL_CORE_FEATURES_CLEAN,
}

# ------------------------------------------------------------
# 41. GENERIC BUILDERS
# ------------------------------------------------------------
def build_summary_df(benchmark_name, benchmark_info):
    rows = []
    phase_boundaries = benchmark_info.get("phase_boundaries", [])

    for condition_name, runs_list in benchmark_info["conditions"].items():
        for run_idx, run in enumerate(runs_list):
            row = extract_desc(
                run,
                phase_boundaries=phase_boundaries,
                tag_name=f"{benchmark_name} | {condition_name}"
            )
            row["benchmark"] = benchmark_name
            row["condition"] = condition_name
            row["benchmark_condition"] = f"{benchmark_name} | {condition_name}"
            row["run_id"] = f"{benchmark_name}_{condition_name}_{run_idx:03d}"
            rows.append(row)

    return pd.DataFrame(rows)

def build_temporal_df(summary_df, benchmark_name, benchmark_info, rng_seed=2027):
    rng = np.random.default_rng(rng_seed)
    rows = []
    phase_boundaries = benchmark_info.get("phase_boundaries", [])

    for condition_name, runs_list in benchmark_info["conditions"].items():
        df_orig_subset = summary_df[
            (summary_df["benchmark"] == benchmark_name) &
            (summary_df["condition"] == condition_name)
        ].copy()

        for _, row in df_orig_subset.iterrows():
            row_dict = row.to_dict()
            row_dict["temporal_label"] = "original"
            rows.append(row_dict)

        for run in runs_list:
            surrogate_run = make_temporal_surrogate(run, rng)
            surrogate_row = extract_desc(
                surrogate_run,
                phase_boundaries=phase_boundaries,
                tag_name=f"{benchmark_name} | {condition_name} | scrambled"
            )
            surrogate_row["benchmark"] = benchmark_name
            surrogate_row["condition"] = condition_name
            surrogate_row["benchmark_condition"] = f"{benchmark_name} | {condition_name}"
            surrogate_row["temporal_label"] = "scrambled"
            rows.append(surrogate_row)

    return pd.DataFrame(rows)

def build_summary_table(summary_df):
    cols = [
        "final_loss",
        "final_score",
        "param_V",
        "func_V",
        "param_C",
        "func_C",
        "mass_param",
        "mass_func",
        "n_events",
        "n_regimes",
    ]
    cols = [c for c in cols if c in summary_df.columns]

    table = (
        summary_df
        .groupby(["benchmark", "condition"])[cols]
        .agg(["mean", "std"])
        .round(4)
    )
    return table

def build_temporal_null_table(temporal_df, random_state=456):
    rows = []

    for block_name, feature_cols in FEATURE_BLOCKS.items():
        res = evaluate_classifier(
            df=temporal_df,
            feature_cols=feature_cols,
            target_col="temporal_label",
            n_splits=2,
            n_repeats=10,
            random_state=random_state
        )
        res["feature_block"] = block_name
        rows.append(res)

    table = (
        pd.DataFrame(rows)
        .set_index("feature_block")
        .sort_values("balanced_accuracy_mean", ascending=False)
        .round(4)
    )
    return table

def get_table_metric(table, block_name, metric_name="balanced_accuracy_mean"):
    if block_name not in table.index:
        return np.nan
    if metric_name not in table.columns:
        return np.nan
    return float(table.loc[block_name, metric_name])

def summarize_case_for_table_M(case_name, role_name, summary_df, temporal_table):
    score_means = summary_df.groupby("condition")["final_score"].mean() if "final_score" in summary_df.columns else pd.Series(dtype=float)
    funcC_means = summary_df.groupby("condition")["func_C"].mean() if "func_C" in summary_df.columns else pd.Series(dtype=float)
    massf_means = summary_df.groupby("condition")["mass_func"].mean() if "mass_func" in summary_df.columns else pd.Series(dtype=float)

    terminal_bacc = get_table_metric(temporal_table, "Terminal only")
    structural_bacc = get_table_metric(temporal_table, "Structural core (clean)")
    global_struct_bacc = get_table_metric(temporal_table, "Global + structural (clean)")
    full_bacc = get_table_metric(temporal_table, "Terminal + global + structural (clean)")

    row = {
        "case": case_name,
        "role": role_name,
        "conditions": " / ".join(sorted(summary_df["condition"].unique())),
        "n_runs": int(len(summary_df)),
        "score_gap_between_conditions": float(score_means.max() - score_means.min()) if len(score_means) >= 2 else np.nan,
        "func_C_gap_between_conditions": float(funcC_means.max() - funcC_means.min()) if len(funcC_means) >= 2 else np.nan,
        "mass_func_gap_between_conditions": float(massf_means.max() - massf_means.min()) if len(massf_means) >= 2 else np.nan,
        "temporal_bacc_terminal_only": terminal_bacc,
        "temporal_bacc_structural_core": structural_bacc,
        "temporal_bacc_global_plus_structural": global_struct_bacc,
        "temporal_bacc_full_block": full_bacc,
        "gain_global_plus_structural_vs_terminal": (
            global_struct_bacc - terminal_bacc
            if np.isfinite(global_struct_bacc) and np.isfinite(terminal_bacc) else np.nan
        ),
        "gain_full_vs_terminal": (
            full_bacc - terminal_bacc
            if np.isfinite(full_bacc) and np.isfinite(terminal_bacc) else np.nan
        ),
    }
    return row

# ------------------------------------------------------------
# 42. TABLE G — AIRFOIL SUMMARY
# ------------------------------------------------------------
airfoil_summary_df = build_summary_df(
    "Airfoil-Self-Noise",
    COMPACT_BENCHMARKS["Airfoil-Self-Noise"]
)

table_G = build_summary_table(airfoil_summary_df)

print("\n=== TABLE G: EXTERNAL BENCHMARK SUMMARY (AIRFOIL ONLY) ===")
print(table_G)

# ------------------------------------------------------------
# 43. TABLE I — AIRFOIL TEMPORAL NULL CONTROL
# ------------------------------------------------------------
airfoil_temporal_df = build_temporal_df(
    airfoil_summary_df,
    "Airfoil-Self-Noise",
    COMPACT_BENCHMARKS["Airfoil-Self-Noise"],
    rng_seed=2027
)

table_I = build_temporal_null_table(airfoil_temporal_df, random_state=456)

print("\n=== TABLE I: EXTERNAL TEMPORAL NULL CONTROL (AIRFOIL ONLY) ===")
print(table_I[[
    "accuracy_mean",
    "accuracy_std",
    "balanced_accuracy_mean",
    "balanced_accuracy_std",
    "macro_f1_mean",
    "macro_f1_std",
    "n_features"
]])

# ------------------------------------------------------------
# 44. TABLE J — MINI-PDEBENCH DARCY SUMMARY
# ------------------------------------------------------------
darcy_summary_df = build_summary_df(
    "Mini-PDEBench-Darcy",
    COMPACT_BENCHMARKS["Mini-PDEBench-Darcy"]
)

table_J = build_summary_table(darcy_summary_df)

print("\n=== TABLE J: MINI-PDEBENCH DARCY SUMMARY ===")
print(table_J)

# ------------------------------------------------------------
# 45. TABLE L — MINI-PDEBENCH DARCY TEMPORAL NULL CONTROL
# ------------------------------------------------------------
darcy_temporal_df = build_temporal_df(
    darcy_summary_df,
    "Mini-PDEBench-Darcy",
    COMPACT_BENCHMARKS["Mini-PDEBench-Darcy"],
    rng_seed=2028
)

table_L = build_temporal_null_table(darcy_temporal_df, random_state=456)

print("\n=== TABLE L: MINI-PDEBENCH DARCY TEMPORAL NULL CONTROL ===")
print(table_L[[
    "accuracy_mean",
    "accuracy_std",
    "balanced_accuracy_mean",
    "balanced_accuracy_std",
    "macro_f1_mean",
    "macro_f1_std",
    "n_features"
]])

# ------------------------------------------------------------
# 46. TABLE M — COMPACT CROSS-CASE EVIDENCE SUMMARY
# ------------------------------------------------------------
table_M = pd.DataFrame([
    summarize_case_for_table_M(
        case_name="Airfoil-Self-Noise",
        role_name="Real external validation",
        summary_df=airfoil_summary_df,
        temporal_table=table_I
    ),
    summarize_case_for_table_M(
        case_name="Mini-PDEBench-Darcy",
        role_name="PDE-based scientific ML validation",
        summary_df=darcy_summary_df,
        temporal_table=table_L
    ),
]).set_index("case").round(4)

print("\n=== TABLE M: COMPACT CROSS-CASE EVIDENCE SUMMARY ===")
print(table_M)

# ------------------------------------------------------------
# 47. STORE FINAL OBJECTS
# ------------------------------------------------------------
SECTION_7C_FINAL_TABLES = {
    "table_G": table_G,
    "table_I": table_I,
    "table_J": table_J,
    "table_L": table_L,
    "table_M": table_M,
    "airfoil_summary_df": airfoil_summary_df,
    "airfoil_temporal_df": airfoil_temporal_df,
    "darcy_summary_df": darcy_summary_df,
    "darcy_temporal_df": darcy_temporal_df,
}

print("\n[SECTION 7C-FINAL] Compact Airfoil + Mini-Darcy layer completed.")
print("Expected main-text tables: G, I, J, L, M")

Using device: cuda
Using cached file: ./external_public_datasets/airfoil_self_noise.dat
Using cached file: ./external_public_datasets/2D_DarcyFlow_beta1.0_Train.hdf5
Available HDF5 keys: ['nu', 'tensor', 'x-coordinate', 'y-coordinate']
Loaded Darcy subset: X=(96, 1024), y=(96, 1024)
Spatial downsample step: 4

Loaded datasets:
Airfoil: (1503, 5) (1503, 1)
Mini-Darcy: (96, 1024) (96, 1024)
Training condition=SGD-constant, seed=123
Training condition=SGD-constant, seed=124
Training condition=SGD-constant, seed=125
Training condition=SGD-constant, seed=126
Training condition=Adam, seed=123
Training condition=Adam, seed=124
Training condition=Adam, seed=125
Training condition=Adam, seed=126
Training condition=SGD-constant, seed=123
Training condition=SGD-constant, seed=124
Training condition=SGD-constant, seed=125
Training condition=SGD-constant, seed=126
Training condition=Adam, seed=123
Training condition=Adam, seed=124
Training condition=Adam, seed=125
Training condition=Adam, seed=126
